## Step35A. Frozen Input and Drug-Perturbation Resource Audit

Step35 tests whether small-molecule transcriptional perturbations can mimic the
convergent five-regulator virtual-suppression response established in Step33 and
oppose the frozen MS3 high-risk program.

No TF, malignant state, GRN, or upstream biological signature will be reselected.

The initial module performs only provenance, frozen-input, environment, and
L1000FWD resource auditing. No formal drug ranking is performed in this module.

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# Cell 1 — Step35 setup

from pathlib import Path
import json
import pandas as pd
import numpy as np
import sys
import platform
import re

print("=" * 90)
print("STEP35A — FROZEN DRUG-REVERSAL SIGNATURE AUDIT")
print("=" * 90)

TCGA_ROOT = Path(_hcc_path(''))

STEP35 = (
    TCGA_ROOT /
    "LIHC_scRNA_Step35_Virtual_Drug_Reversal"
)

DATA = STEP35 / "Data"
FIGURES = STEP35 / "Figures"
LOGS = STEP35 / "Logs"
REFERENCE = STEP35 / "Reference"
RESULTS = STEP35 / "Results"

for d in [
    DATA,
    FIGURES,
    LOGS,
    REFERENCE,
    RESULTS,
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )


STEP25 = (
    TCGA_ROOT /
    "LIHC_scRNA_Step25_CrossPatient_Robustness"
)

STEP26 = (
    TCGA_ROOT /
    "LIHC_scRNA_Step26_MS3_Biology"
)

STEP33 = (
    TCGA_ROOT /
    "LIHC_scRNA_Step33_Virtual_Knockout"
)

STEP34 = (
    TCGA_ROOT /
    "LIHC_scRNA_Step34_Virtual_State_Transition"
)


for p in [
    STEP25,
    STEP26,
    STEP33,
    STEP34,
]:

    assert p.exists(), (
        f"Missing source folder: {p}"
    )


environment = {
    "python":
        sys.version,

    "platform":
        platform.platform(),

    "tcga_root":
        str(TCGA_ROOT),

    "step35":
        str(STEP35),

    "source_steps": {
        "Step25":
            str(STEP25),

        "Step26":
            str(STEP26),

        "Step33":
            str(STEP33),

        "Step34":
            str(STEP34),
    }
}


with open(
    LOGS /
    "Step35A_Environment_and_SourceRoots.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        environment,
        f,
        indent=2,
        ensure_ascii=False
    )


print("Step35 folder:")
print(STEP35)

print()
print("Python:")
print(sys.version.split()[0])

print()
print("=" * 90)
print("Cell 1 complete")
print("=" * 90)

In [ ]:
# Cell 2 — Search frozen signature definitions

print("=" * 90)
print("Cell 2 — Frozen signature source search")
print("=" * 90)

SEARCH_ROOTS = [
    STEP25,
    STEP26,
    STEP33,
]

KEYWORDS = [
    "Core50",
    "Core100",

    "G2M",
    "Mitotic_Entry",
    "Mitotic Entry",

    "Spindle_Chromosome",
    "Spindle",
    "Chromosome",

    "Hepatic_Differentiation",
    "Hepatic Differentiation",

    "Bile_Acid_Metabolism",
    "Bile Acid",
]

TEXT_EXTENSIONS = {
    ".json",
    ".csv",
    ".tsv",
    ".txt",
    ".md",
    ".py",
    ".yaml",
    ".yml",
}

MAX_SIZE_MB = 10

hits = []

for root in SEARCH_ROOTS:

    for path in root.rglob("*"):

        if (
            not path.is_file()
            or
            path.suffix.lower()
            not in TEXT_EXTENSIONS
        ):
            continue

        size_mb = (
            path.stat().st_size
            /
            1024**2
        )

        if size_mb > MAX_SIZE_MB:
            continue

        try:

            text_data = path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

        except Exception:
            continue

        matched = [
            kw
            for kw in KEYWORDS
            if kw.lower()
            in text_data.lower()
            or
            kw.lower()
            in path.name.lower()
        ]

        if matched:

            hits.append({
                "step":
                    root.name,

                "file":
                    path.name,

                "path":
                    str(path),

                "size_kb":
                    round(
                        path.stat().st_size
                        / 1024,
                        2
                    ),

                "matched_keywords":
                    ";".join(
                        matched
                    ),
            })


hits_df = (
    pd.DataFrame(hits)
    .sort_values(
        [
            "step",
            "file"
        ]
    )
    .reset_index(drop=True)
)


hits_df.to_csv(
    RESULTS /
    "Step35A_FrozenSignature_SourceSearch.csv",
    index=False
)


print(
    hits_df.to_string(
        index=False
    )
)

print()
print(
    "Candidate source files:",
    len(hits_df)
)

print()
print("=" * 90)
print("Cell 2 complete")
print("=" * 90)

In [ ]:
# Cell 3 — Prioritize likely endpoint-definition files

print("=" * 90)
print("Cell 3 — Endpoint-definition candidates")
print("=" * 90)

priority_terms = [
    "endpoint",
    "program",
    "definition",
    "gene",
    "core50",
    "lock",
    "frozen",
]

priority = hits_df.copy()

priority[
    "priority_score"
] = priority[
    "file"
].str.lower().apply(
    lambda x:
        sum(
            term in x
            for term in priority_terms
        )
)


priority = (
    priority
    .sort_values(
        [
            "priority_score",
            "step",
            "file"
        ],
        ascending=[
            False,
            True,
            True
        ]
    )
)


print(
    priority[
        [
            "step",
            "file",
            "matched_keywords",
            "priority_score",
            "path"
        ]
    ]
    .head(40)
    .to_string(
        index=False
    )
)


priority.to_csv(
    RESULTS /
    "Step35A_PrioritizedSignatureSources.csv",
    index=False
)

print()
print("=" * 90)
print("Cell 3 complete")
print("=" * 90)

In [ ]:
# Cell 4 — Inspect structured JSON gene-set definitions

print("=" * 90)
print("Cell 4 — Structured frozen gene-set audit")
print("=" * 90)

TARGET_LABELS = [
    "Core50",
    "G2M",
    "Mitotic",
    "Spindle",
    "Hepatic",
    "Bile",
]


def walk_json(
    obj,
    path=""
):

    rows = []

    if isinstance(
        obj,
        dict
    ):

        for k, v in obj.items():

            new_path = (
                f"{path}.{k}"
                if path
                else str(k)
            )

            rows.extend(
                walk_json(
                    v,
                    new_path
                )
            )

    elif isinstance(
        obj,
        list
    ):

        # possible gene-symbol list
        if (
            len(obj) > 0
            and
            all(
                isinstance(x, str)
                for x in obj
            )
        ):

            rows.append({
                "json_path":
                    path,

                "n_items":
                    len(obj),

                "preview":
                    ";".join(
                        obj[:15]
                    ),
            })

        else:

            for i, v in enumerate(obj):

                rows.extend(
                    walk_json(
                        v,
                        f"{path}[{i}]"
                    )
                )

    return rows


json_rows = []

json_paths = list(
    STEP25.rglob("*.json")
) + list(
    STEP26.rglob("*.json")
) + list(
    STEP33.rglob("*.json")
)


for path in json_paths:

    if (
        path.stat().st_size
        >
        10 * 1024**2
    ):
        continue

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            obj = json.load(f)

    except Exception:
        continue


    found = walk_json(obj)

    for row in found:

        searchable = (
            row["json_path"]
            +
            " "
            +
            row["preview"]
        ).lower()

        if any(
            label.lower()
            in searchable
            for label
            in TARGET_LABELS
        ):

            json_rows.append({
                "file":
                    path.name,

                "path":
                    str(path),

                **row
            })


json_audit = pd.DataFrame(
    json_rows
)

if len(json_audit):

    json_audit = (
        json_audit
        .sort_values(
            [
                "file",
                "json_path"
            ]
        )
        .reset_index(drop=True)
    )

    print(
        json_audit.to_string(
            index=False
        )
    )

else:

    print(
        "No directly structured frozen gene-list "
        "definitions found in JSON."
    )


json_audit.to_csv(
    RESULTS /
    "Step35A_JSON_GeneList_Audit.csv",
    index=False
)


print()
print("=" * 90)
print("Cell 4 complete")
print("=" * 90)

In [ ]:
# Cell 5 — Inspect candidate CSV gene lists

print("=" * 90)
print("Cell 5 — Candidate CSV gene-set audit")
print("=" * 90)

csv_rows = []

candidate_csvs = []

for root in [
    STEP25,
    STEP26,
    STEP33,
]:

    candidate_csvs.extend(
        list(
            root.rglob("*.csv")
        )
    )


for path in candidate_csvs:

    if (
        path.stat().st_size
        >
        20 * 1024**2
    ):
        continue

    try:

        df = pd.read_csv(
            path,
            nrows=10
        )

    except Exception:
        continue


    columns_text = (
        " ".join(
            map(
                str,
                df.columns
            )
        )
        .lower()
    )

    name_text = (
        path.name.lower()
    )

    if any(
        kw.lower()
        in (
            columns_text
            +
            " "
            +
            name_text
        )
        for kw
        in KEYWORDS
    ):

        csv_rows.append({
            "file":
                path.name,

            "path":
                str(path),

            "shape_preview_rows":
                len(df),

            "columns":
                ";".join(
                    map(
                        str,
                        df.columns
                    )
                ),
        })


csv_audit = pd.DataFrame(
    csv_rows
)

if len(csv_audit):

    print(
        csv_audit.to_string(
            index=False
        )
    )

else:

    print(
        "No candidate CSV definition files found."
    )


csv_audit.to_csv(
    RESULTS /
    "Step35A_CSV_GeneList_Audit.csv",
    index=False
)

print()
print("=" * 90)
print("Cell 5 complete")
print("=" * 90)

In [ ]:
# Cell 6 — Step35A REPORT

print("=" * 90)
print("STEP35A — FROZEN DRUG-REVERSAL SIGNATURE AUDIT REPORT")
print("=" * 90)

print(
    "\nCandidate text sources:",
    len(hits_df)
)

print(
    "Structured JSON gene-list candidates:",
    len(json_audit)
)

print(
    "CSV gene-list candidates:",
    len(csv_audit)
)


print(
    "\nRequired frozen Step35 signatures:"
)

required = [
    "MS3 Core50",
    "G2M independent",
    "Mitotic Entry independent",
    "Spindle / Chromosome independent",
    "Hepatic Differentiation",
    "Bile Acid Metabolism",
]

for x in required:
    print(" -", x)


print(
    "\nPre-specified Step35 queries:"
)

print(
    "Q1 PRIMARY: Core50 reversal"
)

print(
    "Q2 PRIMARY: independent mitotic-program reversal"
)

print(
    "Q3 SECONDARY: bidirectional MS3-to-differentiated-axis reversal"
)


report35a = {
    "status":
        "STEP35A_SOURCE_AUDIT_COMPLETE",

    "candidate_text_sources":
        int(
            len(hits_df)
        ),

    "structured_json_gene_lists":
        int(
            len(json_audit)
        ),

    "candidate_csv_gene_lists":
        int(
            len(csv_audit)
        ),

    "required_signatures":
        required,

    "formal_query_hierarchy": {
        "Q1":
            "Core50 reversal — primary",

        "Q2":
            (
                "G2M + Mitotic Entry + "
                "Spindle/Chromosome reversal — primary"
            ),

        "Q3":
            (
                "Core50 down + hepatic/bile-acid restoration "
                "— secondary"
            ),
    },

    "drug_resource_plan": {
        "discovery":
            "LINCS L1000FWD",

        "algorithmic_confirmation":
            "L1000CDS2",

        "liver_context":
            [
                "HEPG2",
                "PHH"
            ],
    },

    "claim_boundary":
        (
            "Transcriptomic drug-reversal prediction only; "
            "not therapeutic efficacy, clinical benefit, "
            "or experimentally validated drug sensitivity."
        )
}


REPORT_PATH = (
    LOGS /
    "Step35A_FINAL_REPORT.json"
)

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report35a,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("Saved:")
print(REPORT_PATH)

print()
print("=" * 90)
print("STEP35A STATUS: COMPLETE")
print("=" * 90)

## Step35B. Frozen Drug-Reversal Signature Lock

Drug-reversal queries are constructed exclusively from previously frozen signatures.

The complete Step25 MS3 Conserved Core50 is used for transcriptomic reversal rather
than the CellOracle network-covered Core50 projection.

Independent G2M, Mitotic Entry, Spindle/Chromosome, Hepatic Differentiation and
Bile Acid Metabolism gene sets are recovered from the frozen Step33 endpoint-definition
table.

No gene is selected, removed, ranked or modified using drug-response information.

In [ ]:
# Cell 7 — Load canonical frozen signature sources

print("=" * 90)
print("Cell 7 — Canonical frozen signature sources")
print("=" * 90)

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import shutil
import re

CORE50_SOURCE = (
    STEP25 /
    "Reference" /
    "Step25_FROZEN_MS3_Core50_Genes.txt"
)

ENDPOINT_SOURCE = (
    STEP33 /
    "Reference" /
    "Step33_FROZEN_ENDPOINT_GENESETS.csv"
)

assert CORE50_SOURCE.exists(), CORE50_SOURCE
assert ENDPOINT_SOURCE.exists(), ENDPOINT_SOURCE


# ============================================================
# Preserve source snapshots in Step35 Reference
# ============================================================

CORE50_COPY = (
    REFERENCE /
    CORE50_SOURCE.name
)

ENDPOINT_COPY = (
    REFERENCE /
    ENDPOINT_SOURCE.name
)

shutil.copy2(
    CORE50_SOURCE,
    CORE50_COPY
)

shutil.copy2(
    ENDPOINT_SOURCE,
    ENDPOINT_COPY
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


print("Core50 source:")
print(CORE50_SOURCE)

print("\nEndpoint source:")
print(ENDPOINT_SOURCE)

print("\nSHA256:")
print(
    "Core50  :",
    sha256_file(CORE50_SOURCE)
)

print(
    "Endpoint:",
    sha256_file(ENDPOINT_SOURCE)
)


# ============================================================
# Inspect endpoint schema
# ============================================================

endpoint_raw = pd.read_csv(
    ENDPOINT_SOURCE
)

print("\nEndpoint table shape:")
print(endpoint_raw.shape)

print("\nColumns:")
print(
    endpoint_raw.columns.tolist()
)

print("\nPreview:")
print(
    endpoint_raw.head(15)
    .to_string(index=False)
)

print()
print("=" * 90)
print("Cell 7 complete")
print("=" * 90)

In [ ]:
# Cell 8 — Recover frozen gene sets without reselection

print("=" * 90)
print("Cell 8 — Recover frozen gene sets")
print("=" * 90)


# ============================================================
# Core50
# ============================================================

core50_text = CORE50_SOURCE.read_text(
    encoding="utf-8",
    errors="ignore"
)

core50 = [
    x.strip()
    for x in re.split(
        r"[\n\r,;\t]+",
        core50_text
    )
    if x.strip()
]

# preserve first occurrence / original order
core50 = list(
    dict.fromkeys(core50)
)

print(
    "Core50 genes:",
    len(core50)
)

assert len(core50) == 50, (
    f"Expected frozen Core50=50, got {len(core50)}"
)


# ============================================================
# Detect long-format endpoint columns
# ============================================================

cols_lower = {
    str(c).lower():
        c
    for c in endpoint_raw.columns
}


program_candidates = [
    c
    for c in endpoint_raw.columns
    if any(
        term in str(c).lower()
        for term in [
            "endpoint",
            "program",
            "signature",
            "geneset",
            "gene_set",
            "set_name",
        ]
    )
]

gene_candidates = [
    c
    for c in endpoint_raw.columns
    if (
        str(c).lower()
        in {
            "gene",
            "gene_symbol",
            "symbol",
            "genes"
        }
        or
        (
            "gene" in str(c).lower()
            and
            "count" not in str(c).lower()
            and
            str(c).lower() != "n_genes"
        )
    )
]


print("\nProgram-column candidates:")
print(program_candidates)

print("\nGene-column candidates:")
print(gene_candidates)


assert len(program_candidates) >= 1, (
    "Could not identify endpoint/program column."
)

assert len(gene_candidates) >= 1, (
    "Could not identify gene-symbol column."
)


PROGRAM_COL = program_candidates[0]
GENE_COL = gene_candidates[0]

print(
    "\nUsing:",
    PROGRAM_COL,
    "/",
    GENE_COL
)


# ============================================================
# Normalize table
# ============================================================

endpoint_long = (
    endpoint_raw[
        [
            PROGRAM_COL,
            GENE_COL
        ]
    ]
    .copy()
)

endpoint_long.columns = [
    "signature",
    "gene"
]

endpoint_long[
    "signature"
] = (
    endpoint_long[
        "signature"
    ]
    .astype(str)
    .str.strip()
)

endpoint_long[
    "gene"
] = (
    endpoint_long[
        "gene"
    ]
    .astype(str)
    .str.strip()
)

endpoint_long = endpoint_long.loc[
    (~endpoint_long["gene"].isin(
        [
            "",
            "nan",
            "None"
        ]
    ))
].copy()


print("\nFrozen endpoint labels:")

print(
    endpoint_long[
        "signature"
    ]
    .value_counts()
    .to_string()
)


# ============================================================
# Required exact frozen signatures
# ============================================================

REQUIRED_ENDPOINTS = [
    "G2M_Independent",
    "Mitotic_Entry_Independent",
    "Spindle_Chromosome_Independent",
    "Hepatic_Differentiation",
    "Bile_Acid_Metabolism",
]

available = set(
    endpoint_long[
        "signature"
    ].unique()
)

missing = [
    x
    for x in REQUIRED_ENDPOINTS
    if x not in available
]

assert len(missing) == 0, (
    f"Missing frozen endpoints: {missing}"
)


def frozen_genes(name):

    x = (
        endpoint_long.loc[
            endpoint_long[
                "signature"
            ] == name,
            "gene"
        ]
        .tolist()
    )

    return list(
        dict.fromkeys(x)
    )


g2m = frozen_genes(
    "G2M_Independent"
)

mitotic = frozen_genes(
    "Mitotic_Entry_Independent"
)

spindle = frozen_genes(
    "Spindle_Chromosome_Independent"
)

hepatic = frozen_genes(
    "Hepatic_Differentiation"
)

bile = frozen_genes(
    "Bile_Acid_Metabolism"
)


frozen_sets = {
    "MS3_Core50":
        core50,

    "G2M_Independent":
        g2m,

    "Mitotic_Entry_Independent":
        mitotic,

    "Spindle_Chromosome_Independent":
        spindle,

    "Hepatic_Differentiation":
        hepatic,

    "Bile_Acid_Metabolism":
        bile,
}


print("\nRecovered frozen gene-set sizes:")

for k, v in frozen_sets.items():
    print(
        f"{k:35s}",
        len(v)
    )


# Expected frozen sizes
EXPECTED = {
    "MS3_Core50":
        50,

    "G2M_Independent":
        22,

    "Mitotic_Entry_Independent":
        16,

    "Spindle_Chromosome_Independent":
        29,

    "Hepatic_Differentiation":
        19,

    "Bile_Acid_Metabolism":
        18,
}

for k, expected in EXPECTED.items():

    assert len(
        frozen_sets[k]
    ) == expected, (
        f"{k}: expected {expected}, "
        f"got {len(frozen_sets[k])}"
    )


print()
print("=" * 90)
print("Cell 8 complete")
print("=" * 90)

In [ ]:
# Cell 9 — Lock Step35 drug-reversal queries

print("=" * 90)
print("Cell 9 — Step35 frozen drug-reversal query lock")
print("=" * 90)


# ============================================================
# Q1 — primary conserved MS3 identity
# ============================================================

Q1_UP = list(
    core50
)

Q1_DOWN = []


# ============================================================
# Q2 — primary independent mitotic architecture
# union only for API submission;
# source programs remain individually frozen
# ============================================================

Q2_UP = list(
    dict.fromkeys(
        g2m
        +
        mitotic
        +
        spindle
    )
)

Q2_DOWN = []


# ============================================================
# Q3 — secondary bidirectional state-axis reversal
#
# disease-like:
# Core50 UP
# differentiated biology DOWN
#
# desired drug reversal:
# Core50 suppression +
# differentiated-program restoration
# ============================================================

Q3_UP = list(
    core50
)

Q3_DOWN = list(
    dict.fromkeys(
        hepatic
        +
        bile
    )
)


# ============================================================
# Program overlap audit
# ============================================================

def overlap(a, b):

    return sorted(
        set(a)
        &
        set(b)
    )


overlap_rows = []

names = list(
    frozen_sets.keys()
)

for i in range(len(names)):

    for j in range(
        i + 1,
        len(names)
    ):

        a = names[i]
        b = names[j]

        ov = overlap(
            frozen_sets[a],
            frozen_sets[b]
        )

        overlap_rows.append({
            "signature_A":
                a,

            "signature_B":
                b,

            "n_overlap":
                len(ov),

            "overlap_genes":
                ";".join(ov),
        })


overlap_df = pd.DataFrame(
    overlap_rows
)

overlap_df.to_csv(
    RESULTS /
    "Step35B_FrozenSignature_OverlapAudit.csv",
    index=False
)


# ============================================================
# Save long-format frozen definition
# ============================================================

long_rows = []

for signature, genes in frozen_sets.items():

    for rank, gene in enumerate(
        genes,
        start=1
    ):

        long_rows.append({
            "signature":
                signature,

            "gene":
                gene,

            "frozen_order":
                rank,
        })


frozen_long = pd.DataFrame(
    long_rows
)

frozen_long.to_csv(
    REFERENCE /
    "Step35B_FROZEN_DRUG_REVERSAL_GENESETS.csv",
    index=False
)


# ============================================================
# Save query definitions
# ============================================================

query_rows = []

for query_id, role, up, down in [
    (
        "Q1",
        "PRIMARY",
        Q1_UP,
        Q1_DOWN
    ),
    (
        "Q2",
        "PRIMARY",
        Q2_UP,
        Q2_DOWN
    ),
    (
        "Q3",
        "SECONDARY",
        Q3_UP,
        Q3_DOWN
    ),
]:

    query_rows.append({
        "query_id":
            query_id,

        "role":
            role,

        "n_up":
            len(up),

        "n_down":
            len(down),

        "up_genes":
            ";".join(up),

        "down_genes":
            ";".join(down),
    })


query_df = pd.DataFrame(
    query_rows
)

query_df.to_csv(
    RESULTS /
    "Step35B_FROZEN_QueryDefinitions.csv",
    index=False
)


print("Q1:")
print(
    " UP =", len(Q1_UP),
    " DOWN =", len(Q1_DOWN)
)

print("\nQ2:")
print(
    " UP =", len(Q2_UP),
    " DOWN =", len(Q2_DOWN)
)

print("\nQ3:")
print(
    " UP =", len(Q3_UP),
    " DOWN =", len(Q3_DOWN)
)

print()
print("=" * 90)
print("Cell 9 complete")
print("=" * 90)

In [ ]:
# Cell 10 — Step35B REPORT

print("=" * 90)
print("STEP35B — FROZEN DRUG-REVERSAL SIGNATURE LOCK")
print("=" * 90)

print("\nFrozen signature sizes:")

for k, v in frozen_sets.items():

    print(
        f"{k:35s}: {len(v)}"
    )


print("\nFormal drug-reversal queries:")

print(
    f"Q1 PRIMARY   : "
    f"Core50 UP={len(Q1_UP)}, "
    f"DOWN={len(Q1_DOWN)}"
)

print(
    f"Q2 PRIMARY   : "
    f"Mitotic union UP={len(Q2_UP)}, "
    f"DOWN={len(Q2_DOWN)}"
)

print(
    f"Q3 SECONDARY : "
    f"Core50 UP={len(Q3_UP)}, "
    f"Differentiated DOWN={len(Q3_DOWN)}"
)


print(
    "\nCore50 source:"
)

print(
    "Step25 full frozen Core50 "
    "(NOT Step33 network-covered projection)"
)

print(
    "\nDrug-response-based gene reselection:",
    False
)

print(
    "Drug-response-based thresholding:",
    False
)

print(
    "Program redefinition:",
    False
)


lock35b = {
    "status":
        "STEP35B_FROZEN",

    "core50_source":
        str(CORE50_SOURCE),

    "endpoint_source":
        str(ENDPOINT_SOURCE),

    "source_sha256": {
        "Core50":
            sha256_file(
                CORE50_SOURCE
            ),

        "FrozenEndpoints":
            sha256_file(
                ENDPOINT_SOURCE
            ),
    },

    "signature_sizes": {
        k:
            len(v)

        for k, v
        in frozen_sets.items()
    },

    "queries": {
        "Q1": {
            "role":
                "PRIMARY",

            "definition":
                "MS3 Conserved Core50 reversal",

            "n_up":
                len(Q1_UP),

            "n_down":
                len(Q1_DOWN),
        },

        "Q2": {
            "role":
                "PRIMARY",

            "definition":
                (
                    "Independent G2M + Mitotic Entry + "
                    "Spindle/Chromosome reversal"
                ),

            "n_up":
                len(Q2_UP),

            "n_down":
                len(Q2_DOWN),
        },

        "Q3": {
            "role":
                "SECONDARY",

            "definition":
                (
                    "Core50 suppression plus "
                    "hepatic/bile-acid restoration"
                ),

            "n_up":
                len(Q3_UP),

            "n_down":
                len(Q3_DOWN),
        },
    },

    "reselection":
        False,

    "claim_boundary":
        (
            "Frozen transcriptomic disease-state "
            "signatures only; no drug-response-derived "
            "gene selection."
        )
}


LOCK_PATH = (
    LOGS /
    "Step35B_FROZEN_SIGNATURE_LOCK.json"
)

with open(
    LOCK_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        lock35b,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("Saved:")
print(
    REFERENCE /
    "Step35B_FROZEN_DRUG_REVERSAL_GENESETS.csv"
)

print(
    RESULTS /
    "Step35B_FROZEN_QueryDefinitions.csv"
)

print(LOCK_PATH)

print()
print("=" * 90)
print("STEP35B STATUS: FROZEN")
print("=" * 90)

## Step35C. L1000FWD Transcriptomic Drug-Reversal Discovery

Three pre-specified frozen queries are submitted to L1000FWD.

Formal reversal evidence uses only the returned opposite drug-induced signatures.

Q1 and Q2 are primary discovery queries.
Q3 is secondary bidirectional-state-axis support.

No drug is selected, excluded, or re-ranked before all three frozen queries are completed.

In [ ]:
# Cell 11 — L1000FWD API setup

print("=" * 90)
print("Cell 11 — L1000FWD API setup")
print("=" * 90)

import requests
import json
import time
import pandas as pd
import numpy as np

L1000FWD_URL = (
    "https://maayanlab.cloud/l1000fwd/"
)

QUERY_FILE = (
    RESULTS /
    "Step35B_FROZEN_QueryDefinitions.csv"
)

assert QUERY_FILE.exists()

query_def = pd.read_csv(
    QUERY_FILE
)

print(
    query_def[
        [
            "query_id",
            "role",
            "n_up",
            "n_down"
        ]
    ].to_string(index=False)
)


def split_genes(x):

    if pd.isna(x):
        return []

    x = str(x).strip()

    if x == "":
        return []

    return [
        g.strip().upper()
        for g in x.split(";")
        if g.strip()
    ]


queries35 = {}

for _, row in query_def.iterrows():

    qid = row["query_id"]

    queries35[qid] = {
        "role":
            row["role"],

        "up_genes":
            split_genes(
                row["up_genes"]
            ),

        "down_genes":
            split_genes(
                row["down_genes"]
            ),
    }


for qid, q in queries35.items():

    print(
        f"{qid}: "
        f"UP={len(q['up_genes'])}, "
        f"DOWN={len(q['down_genes'])}, "
        f"ROLE={q['role']}"
    )


assert len(
    queries35["Q1"]["up_genes"]
) == 50

assert len(
    queries35["Q2"]["up_genes"]
) == 58

assert len(
    queries35["Q3"]["up_genes"]
) == 50

assert len(
    queries35["Q3"]["down_genes"]
) == 36

print()
print("=" * 90)
print("Cell 11 complete")
print("=" * 90)

In [ ]:
# Cell 12 — Submit frozen queries to L1000FWD

print("=" * 90)
print("Cell 12 — Submit frozen L1000FWD queries")
print("=" * 90)

session = requests.Session()

query_ids = {}
submission_audit = []


for qid in [
    "Q1",
    "Q2",
    "Q3"
]:

    q = queries35[qid]

    payload = {
        "up_genes":
            q["up_genes"],

        "down_genes":
            q["down_genes"],
    }


    # Save exact submitted payload
    payload_path = (
        REFERENCE /
        f"Step35C_{qid}_L1000FWD_SubmittedPayload.json"
    )

    with open(
        payload_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            ensure_ascii=False
        )


    print()
    print(
        f"Submitting {qid}: "
        f"UP={len(payload['up_genes'])}, "
        f"DOWN={len(payload['down_genes'])}"
    )


    response = session.post(
        L1000FWD_URL + "sig_search",
        json=payload,
        timeout=120
    )


    print(
        "HTTP:",
        response.status_code
    )


    if response.status_code != 200:

        print(
            response.text[:1000]
        )

        raise RuntimeError(
            f"{qid} submission failed."
        )


    result = response.json()

    assert (
        "result_id"
        in result
    ), result


    result_id = result[
        "result_id"
    ]

    query_ids[qid] = result_id


    submission_audit.append({
        "query_id":
            qid,

        "role":
            q["role"],

        "n_up":
            len(
                payload["up_genes"]
            ),

        "n_down":
            len(
                payload["down_genes"]
            ),

        "result_id":
            result_id,

        "http_status":
            response.status_code,
    })


    print(
        "result_id:",
        result_id
    )

    time.sleep(0.5)


submission_df = pd.DataFrame(
    submission_audit
)

submission_df.to_csv(
    RESULTS /
    "Step35C_L1000FWD_SubmissionAudit.csv",
    index=False
)


with open(
    LOGS /
    "Step35C_L1000FWD_ResultIDs.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        query_ids,
        f,
        indent=2
    )


print()
print("=" * 90)
print("All three frozen queries submitted successfully.")
print("=" * 90)

In [ ]:
# Cell 13 — Retrieve top-50 opposite signatures

print("=" * 90)
print("Cell 13 — Retrieve L1000FWD top results")
print("=" * 90)

all_opposite = []
all_similar = []

raw_topn = {}


for qid, result_id in query_ids.items():

    url = (
        L1000FWD_URL
        +
        "result/topn/"
        +
        result_id
    )

    r = session.get(
        url,
        timeout=120
    )

    print(
        qid,
        "HTTP:",
        r.status_code
    )

    assert (
        r.status_code == 200
    ), r.text[:1000]


    data = r.json()

    raw_topn[qid] = data


    opposite = data.get(
        "opposite",
        []
    )

    similar = data.get(
        "similar",
        []
    )


    print(
        f"{qid}: "
        f"opposite={len(opposite)}, "
        f"similar={len(similar)}"
    )


    for rank, item in enumerate(
        opposite,
        start=1
    ):

        all_opposite.append({
            "query_id":
                qid,

            "query_role":
                queries35[qid]["role"],

            "rank":
                rank,

            "score":
                float(
                    item["scores"]
                ),

            "sig_id":
                item["sig_id"],
        })


    for rank, item in enumerate(
        similar,
        start=1
    ):

        all_similar.append({
            "query_id":
                qid,

            "query_role":
                queries35[qid]["role"],

            "rank":
                rank,

            "score":
                float(
                    item["scores"]
                ),

            "sig_id":
                item["sig_id"],
        })


with open(
    DATA /
    "Step35C_L1000FWD_RawTopN.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        raw_topn,
        f,
        indent=2
    )


opposite_df = pd.DataFrame(
    all_opposite
)

similar_df = pd.DataFrame(
    all_similar
)


opposite_df.to_csv(
    RESULTS /
    "Step35C_L1000FWD_OppositeSignatures_Raw.csv",
    index=False
)

similar_df.to_csv(
    RESULTS /
    "Step35C_L1000FWD_SimilarSignatures_AuditOnly.csv",
    index=False
)


print()
print(
    "Formal opposite signatures:",
    len(opposite_df)
)

print()
print("=" * 90)
print("Cell 13 complete")
print("=" * 90)

In [ ]:
# Cell 14 — Retrieve metadata for unique opposite signatures

print("=" * 90)
print("Cell 14 — Retrieve signature metadata")
print("=" * 90)

unique_sig_ids = (
    opposite_df[
        "sig_id"
    ]
    .drop_duplicates()
    .tolist()
)

print(
    "Unique opposite signatures:",
    len(unique_sig_ids)
)


metadata_records = []
metadata_failures = []


for n, sig_id in enumerate(
    unique_sig_ids,
    start=1
):

    url = (
        L1000FWD_URL
        +
        "sig/"
        +
        sig_id
    )

    try:

        r = session.get(
            url,
            timeout=120
        )

        if r.status_code != 200:

            metadata_failures.append({
                "sig_id":
                    sig_id,

                "status":
                    r.status_code
            })

            continue


        x = r.json()


        metadata_records.append({
            "sig_id":
                sig_id,

            "pert_desc":
                x.get(
                    "pert_desc"
                ),

            "pert_id":
                x.get(
                    "pert_id"
                ),

            "cell_id":
                x.get(
                    "cell_id"
                ),

            "pert_time":
                x.get(
                    "pert_time"
                ),

            "pert_dose":
                x.get(
                    "pert_dose"
                ),

            "batch":
                x.get(
                    "batch"
                ),

            "pvalue":
                x.get(
                    "pvalue"
                ),

            "mean_cosine_dist_centered_by_batch":
                x.get(
                    "mean_cosine_dist_centered_by_batch"
                ),
        })


    except Exception as e:

        metadata_failures.append({
            "sig_id":
                sig_id,

            "status":
                str(e)
        })


    if (
        n % 25
        == 0
    ):

        print(
            f"Retrieved {n}/"
            f"{len(unique_sig_ids)}"
        )

    time.sleep(0.05)


metadata_df = pd.DataFrame(
    metadata_records
)

failure_df = pd.DataFrame(
    metadata_failures
)


metadata_df.to_csv(
    RESULTS /
    "Step35C_L1000FWD_SignatureMetadata.csv",
    index=False
)

failure_df.to_csv(
    LOGS /
    "Step35C_L1000FWD_MetadataFailures.csv",
    index=False
)


formal_sig = (
    opposite_df
    .merge(
        metadata_df,
        on="sig_id",
        how="left",
        validate="many_to_one"
    )
)


formal_sig.to_csv(
    RESULTS /
    "Step35C_L1000FWD_OppositeSignatures_Annotated.csv",
    index=False
)


print()
print(
    "Metadata retrieved:",
    len(metadata_df)
)

print(
    "Metadata failures:",
    len(failure_df)
)

print()
print("=" * 90)
print("Cell 14 complete")
print("=" * 90)

In [ ]:
# Cell 15 — Step35C REPORT

print("=" * 90)
print("STEP35C — L1000FWD DRUG-REVERSAL DISCOVERY REPORT")
print("=" * 90)


summary_rows = []

for qid in [
    "Q1",
    "Q2",
    "Q3"
]:

    sub = (
        formal_sig.loc[
            formal_sig[
                "query_id"
            ] == qid
        ]
        .sort_values(
            "rank"
        )
    )


    summary_rows.append({
        "query_id":
            qid,

        "role":
            queries35[qid]["role"],

        "n_opposite_signatures":
            len(sub),

        "n_unique_perturbations":
            sub[
                "pert_id"
            ].nunique(
                dropna=True
            ),

        "n_unique_compound_names":
            sub[
                "pert_desc"
            ].nunique(
                dropna=True
            ),

        "n_cell_lines":
            sub[
                "cell_id"
            ].nunique(
                dropna=True
            ),
    })


summary35c = pd.DataFrame(
    summary_rows
)

print("\nQuery summary:\n")

print(
    summary35c.to_string(
        index=False
    )
)


print(
    "\nTop 10 opposite signatures per query:"
)

for qid in [
    "Q1",
    "Q2",
    "Q3"
]:

    print()
    print("-" * 90)
    print(qid)

    sub = (
        formal_sig.loc[
            formal_sig[
                "query_id"
            ] == qid
        ]
        .sort_values(
            "rank"
        )
        .head(10)
    )

    show_cols = [
        "rank",
        "score",
        "pert_desc",
        "pert_id",
        "cell_id",
        "pert_time",
        "pert_dose",
    ]

    print(
        sub[
            show_cols
        ].to_string(
            index=False
        )
    )


summary35c.to_csv(
    RESULTS /
    "Step35C_L1000FWD_QuerySummary.csv",
    index=False
)


report35c = {
    "status":
        "STEP35C_L1000FWD_DISCOVERY_COMPLETE",

    "resource":
        "L1000FWD",

    "formal_direction":
        "opposite",

    "queries": {
        row["query_id"]:
            {
                k: (
                    int(v)
                    if isinstance(
                        v,
                        (np.integer,)
                    )
                    else v
                )
                for k, v
                in row.items()
                if k != "query_id"
            }

        for row in summary_rows
    },

    "metadata_failures":
        int(
            len(failure_df)
        ),

    "formal_use":
        (
            "Signature-level drug-reversal discovery; "
            "not yet final drug prioritization."
        ),

    "next_step":
        (
            "Aggregate signatures to perturbation/drug level "
            "and evaluate Q1-Q2 convergence."
        ),

    "claim_boundary":
        (
            "Transcriptomic reversal prediction from LINCS/L1000; "
            "not therapeutic efficacy."
        )
}


with open(
    LOGS /
    "Step35C_FINAL_REPORT.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report35c,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("=" * 90)
print("STEP35C STATUS: COMPLETE")
print("=" * 90)

## Step35D. Perturbation-Level Cross-Query Convergence

L1000FWD results are aggregated from individual signatures to perturbation identity
using `pert_id` as the canonical key.

Q1 and Q2 define primary transcriptomic reversal evidence.

Q3 is retained only as secondary support and does not contribute to the primary
consensus score.

Because connectivity-score scales differ across queries, cross-query convergence is
based on within-query rank rather than direct averaging of raw scores.

No final therapeutic ranking is assigned at this stage.

In [ ]:
# Cell 16 — Canonicalize L1000FWD signature-level results

print("=" * 90)
print("Cell 16 — Canonicalize signature-level results")
print("=" * 90)

import pandas as pd
import numpy as np
import json

ANNOTATED_PATH = (
    RESULTS /
    "Step35C_L1000FWD_OppositeSignatures_Annotated.csv"
)

assert ANNOTATED_PATH.exists()

sig35 = pd.read_csv(
    ANNOTATED_PATH
)

assert len(sig35) == 150

assert set(
    sig35["query_id"]
) == {
    "Q1",
    "Q2",
    "Q3"
}


INVALID_NAMES = {
    "",
    "-666",
    "nan",
    "none",
    "null",
    "na",
}


def clean_drug_name(x):

    if pd.isna(x):
        return None

    x = str(x).strip()

    if x.lower() in INVALID_NAMES:
        return None

    return x


sig35[
    "drug_name_clean"
] = (
    sig35[
        "pert_desc"
    ]
    .apply(
        clean_drug_name
    )
)


# pert_id is the canonical identity.
sig35[
    "drug_label"
] = np.where(
    sig35[
        "drug_name_clean"
    ].notna(),
    sig35[
        "drug_name_clean"
    ],
    sig35[
        "pert_id"
    ]
)


# ============================================================
# Technical audit
# ============================================================

print(
    "Signature rows:",
    len(sig35)
)

print(
    "Unique pert_ids:",
    sig35[
        "pert_id"
    ].nunique()
)

print(
    "Rows with resolved names:",
    int(
        sig35[
            "drug_name_clean"
        ].notna().sum()
    )
)

print(
    "Rows with unresolved names:",
    int(
        sig35[
            "drug_name_clean"
        ].isna().sum()
    )
)


for qid in [
    "Q1",
    "Q2",
    "Q3"
]:

    sub = sig35.loc[
        sig35[
            "query_id"
        ] == qid
    ]

    assert len(sub) == 50

    assert set(
        sub["rank"].astype(int)
    ) == set(
        range(1, 51)
    )


sig35.to_csv(
    RESULTS /
    "Step35D_OppositeSignatures_Canonicalized.csv",
    index=False
)

print()
print("=" * 90)
print("Cell 16 complete")
print("=" * 90)

In [ ]:
# Cell 17 — Aggregate signatures to perturbation level

print("=" * 90)
print("Cell 17 — Perturbation-level aggregation")
print("=" * 90)


def canonical_name(series):

    vals = [
        str(x).strip()
        for x in series
        if (
            pd.notna(x)
            and
            str(x).strip().lower()
            not in INVALID_NAMES
        )
    ]

    if len(vals) == 0:
        return None

    vc = pd.Series(
        vals
    ).value_counts()

    return vc.index[0]


agg_rows = []


for (
    qid,
    role,
    pert_id
), sub in sig35.groupby(
    [
        "query_id",
        "query_role",
        "pert_id"
    ],
    dropna=False
):

    name = canonical_name(
        sub[
            "drug_name_clean"
        ]
    )

    label = (
        name
        if name is not None
        else pert_id
    )


    agg_rows.append({
        "query_id":
            qid,

        "query_role":
            role,

        "pert_id":
            pert_id,

        "drug_name":
            name,

        "drug_label":
            label,

        "best_rank":
            int(
                sub[
                    "rank"
                ].min()
            ),

        "median_rank":
            float(
                sub[
                    "rank"
                ].median()
            ),

        # More negative = stronger opposite signature
        "best_score":
            float(
                sub[
                    "score"
                ].min()
            ),

        "median_score":
            float(
                sub[
                    "score"
                ].median()
            ),

        "n_signatures":
            int(
                len(sub)
            ),

        "n_cell_lines":
            int(
                sub[
                    "cell_id"
                ].nunique(
                    dropna=True
                )
            ),

        "cell_lines":
            ";".join(
                sorted(
                    sub[
                        "cell_id"
                    ]
                    .dropna()
                    .astype(str)
                    .unique()
                )
            ),

        "n_times":
            int(
                sub[
                    "pert_time"
                ].nunique(
                    dropna=True
                )
            ),

        "times":
            ";".join(
                sorted(
                    sub[
                        "pert_time"
                    ]
                    .dropna()
                    .astype(str)
                    .unique()
                )
            ),

        "n_doses":
            int(
                sub[
                    "pert_dose"
                ].nunique(
                    dropna=True
                )
            ),

        "doses":
            ";".join(
                sorted(
                    sub[
                        "pert_dose"
                    ]
                    .dropna()
                    .astype(str)
                    .unique()
                )
            ),
    })


pert_query = pd.DataFrame(
    agg_rows
)


# ============================================================
# Within-query rank strength
#
# rank 1  -> 1.00
# rank 50 -> 0.02
#
# This avoids comparing raw connectivity magnitudes
# between differently sized/signature queries.
# ============================================================

TOP_N = 50

pert_query[
    "rank_strength"
] = (
    TOP_N
    + 1
    -
    pert_query[
        "best_rank"
    ]
) / TOP_N


pert_query.to_csv(
    RESULTS /
    "Step35D_PerturbationLevel_ByQuery.csv",
    index=False
)


print(
    "Perturbation-query rows:",
    len(pert_query)
)

print()

print(
    pert_query.groupby(
        "query_id"
    )[
        "pert_id"
    ]
    .nunique()
    .to_string()
)

print()
print("=" * 90)
print("Cell 17 complete")
print("=" * 90)

In [ ]:
# Cell 18 — Primary Q1/Q2 cross-query convergence

print("=" * 90)
print("Cell 18 — Primary cross-query convergence")
print("=" * 90)


# ============================================================
# Drug-name map
# ============================================================

name_map = (
    pert_query
    .groupby(
        "pert_id",
        as_index=False
    )
    .agg(
        drug_name=(
            "drug_name",
            canonical_name
        )
    )
)

name_map[
    "drug_label"
] = np.where(
    name_map[
        "drug_name"
    ].notna(),
    name_map[
        "drug_name"
    ],
    name_map[
        "pert_id"
    ]
)


# ============================================================
# Helper pivot
# ============================================================

def wide_metric(metric):

    x = (
        pert_query
        .pivot(
            index="pert_id",
            columns="query_id",
            values=metric
        )
    )

    x.columns = [
        f"{metric}_{c}"
        for c in x.columns
    ]

    return x


metrics = [
    "best_rank",
    "median_rank",
    "best_score",
    "median_score",
    "rank_strength",
    "n_signatures",
    "n_cell_lines",
]


wide_parts = [
    wide_metric(m)
    for m in metrics
]

drug_wide = pd.concat(
    wide_parts,
    axis=1
).reset_index()


drug_wide = drug_wide.merge(
    name_map,
    on="pert_id",
    how="left"
)


# ============================================================
# Presence flags
# ============================================================

for qid in [
    "Q1",
    "Q2",
    "Q3"
]:

    drug_wide[
        f"present_{qid}"
    ] = (
        drug_wide[
            f"best_rank_{qid}"
        ].notna()
    )


drug_wide[
    "primary_convergent"
] = (
    drug_wide[
        "present_Q1"
    ]
    &
    drug_wide[
        "present_Q2"
    ]
)


drug_wide[
    "secondary_Q3_support"
] = (
    drug_wide[
        "present_Q3"
    ]
)


# ============================================================
# Primary consensus rank strength
#
# Only defined when BOTH Q1 and Q2 are present.
# Q3 is NOT included.
# ============================================================

drug_wide[
    "primary_consensus_strength"
] = np.where(
    drug_wide[
        "primary_convergent"
    ],

    (
        drug_wide[
            "rank_strength_Q1"
        ]
        +
        drug_wide[
            "rank_strength_Q2"
        ]
    )
    / 2,

    np.nan
)


drug_wide[
    "primary_rank_sum"
] = np.where(
    drug_wide[
        "primary_convergent"
    ],

    (
        drug_wide[
            "best_rank_Q1"
        ]
        +
        drug_wide[
            "best_rank_Q2"
        ]
    ),

    np.nan
)


# ============================================================
# Evidence tier
# ============================================================

def evidence_tier(row):

    if (
        row["primary_convergent"]
        and
        row["secondary_Q3_support"]
    ):
        return (
            "A_PRIMARY_CONVERGENT_PLUS_Q3"
        )

    if row[
        "primary_convergent"
    ]:
        return (
            "B_PRIMARY_CONVERGENT"
        )

    if (
        (
            row["present_Q1"]
            or
            row["present_Q2"]
        )
        and
        row["secondary_Q3_support"]
    ):
        return (
            "C_SINGLE_PRIMARY_PLUS_Q3"
        )

    return (
        "D_SINGLE_QUERY_OR_OTHER"
    )


drug_wide[
    "evidence_tier"
] = drug_wide.apply(
    evidence_tier,
    axis=1
)


# ============================================================
# Descriptive sorting only
# ============================================================

tier_order = {
    "A_PRIMARY_CONVERGENT_PLUS_Q3":
        0,

    "B_PRIMARY_CONVERGENT":
        1,

    "C_SINGLE_PRIMARY_PLUS_Q3":
        2,

    "D_SINGLE_QUERY_OR_OTHER":
        3,
}

drug_wide[
    "_tier_order"
] = (
    drug_wide[
        "evidence_tier"
    ]
    .map(
        tier_order
    )
)


drug_wide = (
    drug_wide
    .sort_values(
        [
            "_tier_order",
            "primary_consensus_strength",
            "primary_rank_sum",
        ],
        ascending=[
            True,
            False,
            True,
        ]
    )
    .drop(
        columns="_tier_order"
    )
    .reset_index(drop=True)
)


drug_wide.to_csv(
    RESULTS /
    "Step35D_CrossQuery_PerturbationConvergence.csv",
    index=False
)


# ============================================================
# Primary candidate pool
# ============================================================

primary_pool = (
    drug_wide.loc[
        drug_wide[
            "primary_convergent"
        ]
    ]
    .copy()
)

primary_pool.to_csv(
    RESULTS /
    "Step35D_PRIMARY_ConvergentPerturbations.csv",
    index=False
)


print(
    "All unique perturbations:",
    len(drug_wide)
)

print(
    "Primary Q1+Q2 convergent:",
    len(primary_pool)
)

print(
    "Primary Q1+Q2 + Q3 support:",
    int(
        (
            primary_pool[
                "secondary_Q3_support"
            ]
        ).sum()
    )
)

print()
print("=" * 90)
print("Cell 18 complete")
print("=" * 90)

In [ ]:
# Cell 19 — Step35D REPORT

print("=" * 90)
print("STEP35D — DRUG-LEVEL CROSS-QUERY CONVERGENCE REPORT")
print("=" * 90)


n_primary = int(
    drug_wide[
        "primary_convergent"
    ].sum()
)

n_primary_q3 = int(
    (
        drug_wide[
            "primary_convergent"
        ]
        &
        drug_wide[
            "secondary_Q3_support"
        ]
    ).sum()
)


print(
    "\nUnique perturbations across Q1/Q2/Q3:",
    len(drug_wide)
)

print(
    "Q1 + Q2 primary-convergent perturbations:",
    n_primary
)

print(
    "Q1 + Q2 + secondary Q3 support:",
    n_primary_q3
)


print(
    "\nEvidence-tier counts:"
)

print(
    drug_wide[
        "evidence_tier"
    ]
    .value_counts()
    .to_string()
)


# ============================================================
# Top primary-convergent perturbations
# ============================================================

primary_show = (
    drug_wide.loc[
        drug_wide[
            "primary_convergent"
        ]
    ]
    .copy()
)


SHOW_COLS = [
    "drug_label",
    "pert_id",

    "best_rank_Q1",
    "best_rank_Q2",
    "best_rank_Q3",

    "primary_consensus_strength",

    "best_score_Q1",
    "best_score_Q2",
    "best_score_Q3",

    "secondary_Q3_support",

    "n_signatures_Q1",
    "n_signatures_Q2",
    "n_signatures_Q3",

    "n_cell_lines_Q1",
    "n_cell_lines_Q2",
    "n_cell_lines_Q3",

    "evidence_tier",
]


print(
    "\nTop primary-convergent perturbations:"
)

print(
    primary_show[
        SHOW_COLS
    ]
    .head(25)
    .to_string(
        index=False
    )
)


# ============================================================
# Name-resolution audit
# ============================================================

unresolved_primary = (
    primary_show.loc[
        primary_show[
            "drug_name"
        ].isna()
    ][
        [
            "pert_id",
            "best_rank_Q1",
            "best_rank_Q2",
            "best_rank_Q3",
            "primary_consensus_strength",
        ]
    ]
)

unresolved_primary.to_csv(
    RESULTS /
    "Step35D_PRIMARY_UnresolvedPerturbationNames.csv",
    index=False
)


print(
    "\nPrimary-convergent perturbations "
    "with unresolved drug names:",
    len(unresolved_primary)
)


# ============================================================
# Save report
# ============================================================

report35d = {
    "status":
        "STEP35D_PRIMARY_CONVERGENCE_COMPLETE",

    "canonical_identity":
        "pert_id",

    "cross_query_metric":
        (
            "Within-query best-rank strength; "
            "raw scores are not averaged across queries."
        ),

    "unique_perturbations":
        int(
            len(drug_wide)
        ),

    "primary_Q1_Q2_convergent":
        n_primary,

    "primary_Q1_Q2_with_Q3_support":
        n_primary_q3,

    "Q3_in_primary_score":
        False,

    "unresolved_primary_names":
        int(
            len(
                unresolved_primary
            )
        ),

    "formal_interpretation":
        (
            "Perturbations appearing among opposite signatures "
            "for both frozen primary queries constitute the "
            "primary drug-reversal candidate pool."
        ),

    "not_yet_performed": [
        "liver-context prioritization",
        "mechanism-of-action annotation",
        "target concordance",
        "final drug shortlist",
        "therapeutic ranking"
    ],

    "claim_boundary":
        (
            "Cross-query transcriptomic reversal convergence; "
            "not therapeutic efficacy."
        )
}


REPORT35D = (
    LOGS /
    "Step35D_FINAL_REPORT.json"
)

with open(
    REPORT35D,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report35d,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("Saved:")
print(
    RESULTS /
    "Step35D_PRIMARY_ConvergentPerturbations.csv"
)
print(REPORT35D)

print()
print("=" * 90)
print("STEP35D STATUS: COMPLETE")
print("=" * 90)

## Step35E. Drug Annotation and Liver-Context Availability Audit

Primary Q1/Q2-convergent perturbations are annotated using the official L1000FWD
drug metadata indexed by `pert_id`.

HEPG2 and PHH signature availability is audited separately.

Availability of a liver-context signature is not interpreted as evidence of reversal.
Formal liver-context reversal will be evaluated subsequently from the corresponding
drug-induced up/down gene signatures.

In [ ]:
# Cell 20 — Robust download of required L1000FWD metadata

print("=" * 90)
print("Cell 20 — Download required L1000FWD metadata")
print("=" * 90)

from pathlib import Path
import subprocess
import shutil
import hashlib
import pandas as pd


DRUG_META_URL = (
    "https://maayanlab.cloud/"
    "l1000fwd/download/Drugs_metadata.csv"
)

SIG_META_URL = (
    "https://maayanlab.cloud/"
    "l1000fwd/download/CD_signature_metadata.csv"
)

DRUG_META_PATH = (
    REFERENCE /
    "L1000FWD_Drugs_metadata.csv"
)

SIG_META_PATH = (
    REFERENCE /
    "L1000FWD_CD_signature_metadata.csv"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def robust_download(url, path):

    path = Path(path)

    print()
    print("-" * 90)
    print("Target:")
    print(path)
    print("URL:")
    print(url)

    # --------------------------------------------------------
    # If an existing complete-looking file can be read as CSV,
    # reuse it rather than downloading again.
    # --------------------------------------------------------

    if (
        path.exists()
        and
        path.stat().st_size > 100_000
    ):

        try:

            test = pd.read_csv(
                path,
                nrows=5
            )

            print(
                f"Existing valid CSV found: "
                f"{path.stat().st_size / 1024**2:.2f} MB"
            )

            print(
                "Columns:",
                test.columns.tolist()
            )

            return

        except Exception:

            print(
                "Existing file is incomplete/corrupt; "
                "will resume or replace."
            )

    # --------------------------------------------------------
    # Prefer curl
    # --------------------------------------------------------

    curl = shutil.which("curl")

    if curl is None:

        raise RuntimeError(
            "curl not found in WSL. "
            "Install with: sudo apt install curl"
        )

    cmd = [
        curl,

        # Follow redirects
        "-L",

        # Fail on HTTP errors
        "--fail",

        # Resume partial download when possible
        "-C",
        "-",

        # Retry transient failures
        "--retry",
        "20",

        "--retry-delay",
        "3",

        "--retry-max-time",
        "1800",

        "--retry-all-errors",

        # Connection timeout only
        "--connect-timeout",
        "30",

        # Do NOT impose a total transfer time limit
        "--max-time",
        "0",

        # Show progress bar
        "--progress-bar",

        # Output path
        "-o",
        str(path),

        url,
    ]

    print()
    print("Starting/resuming download...")
    print()

    result = subprocess.run(
        cmd,
        check=False
    )

    if result.returncode != 0:

        raise RuntimeError(
            f"curl download failed with "
            f"return code {result.returncode}. "
            f"Re-run this cell to resume."
        )

    # --------------------------------------------------------
    # Validate downloaded CSV
    # --------------------------------------------------------

    assert path.exists(), (
        f"Downloaded file missing: {path}"
    )

    assert path.stat().st_size > 100_000, (
        f"Downloaded file unexpectedly small: {path}"
    )

    try:

        test = pd.read_csv(
            path,
            nrows=5
        )

    except Exception as e:

        raise RuntimeError(
            f"Downloaded file is not a valid CSV: {path}"
        ) from e


    print()
    print(
        f"Download complete: "
        f"{path.stat().st_size / 1024**2:.2f} MB"
    )

    print(
        "Columns:",
        test.columns.tolist()
    )

    print(
        "SHA256:",
        sha256_file(path)
    )


# ============================================================
# Only the two metadata files actually needed
# ============================================================

robust_download(
    DRUG_META_URL,
    DRUG_META_PATH
)

robust_download(
    SIG_META_URL,
    SIG_META_PATH
)


print()
print("=" * 90)
print("Both required metadata files downloaded and validated.")
print("=" * 90)

In [ ]:
# Cell 21 — Inspect official metadata schemas

print("=" * 90)
print("Cell 21 — Metadata schema audit")
print("=" * 90)


drug_meta = pd.read_csv(
    DRUG_META_PATH
)

sig_meta = pd.read_csv(
    SIG_META_PATH
)


print(
    "Drug metadata:",
    drug_meta.shape
)

print(
    "Drug columns:"
)

print(
    drug_meta.columns.tolist()
)


print(
    "\nSignature metadata:",
    sig_meta.shape
)

print(
    "Signature columns:"
)

print(
    sig_meta.columns.tolist()
)


print(
    "\nDrug metadata preview:"
)

print(
    drug_meta.head(5)
    .to_string(
        index=False
    )
)


print(
    "\nSignature metadata preview:"
)

print(
    sig_meta.head(5)
    .to_string(
        index=False
    )
)


# ============================================================
# Normalize column-name map
# ============================================================

drug_colmap = {
    str(c).lower():
        c
    for c
    in drug_meta.columns
}

sig_colmap = {
    str(c).lower():
        c
    for c
    in sig_meta.columns
}


assert (
    "pert_id"
    in drug_colmap
), (
    "Drug metadata missing pert_id."
)

assert (
    "pert_id"
    in sig_colmap
), (
    "Signature metadata missing pert_id."
)

assert (
    "sig_id"
    in sig_colmap
), (
    "Signature metadata missing sig_id."
)

assert (
    "cell_id"
    in sig_colmap
), (
    "Signature metadata missing cell_id."
)


print()
print("=" * 90)
print("Cell 21 complete")
print("=" * 90)

In [ ]:
# Cell 22 — Resolve primary candidates using official metadata

print("=" * 90)
print("Cell 22 — Official drug annotation")
print("=" * 90)


PRIMARY_PATH = (
    RESULTS /
    "Step35D_PRIMARY_ConvergentPerturbations.csv"
)

primary35 = pd.read_csv(
    PRIMARY_PATH
)

assert len(
    primary35
) == 24


# ============================================================
# Detect useful annotation columns
# ============================================================

def find_col(
    columns,
    candidates
):

    lookup = {
        str(c).lower():
            c
        for c
        in columns
    }

    for x in candidates:

        if (
            x.lower()
            in lookup
        ):

            return lookup[
                x.lower()
            ]

    return None


DRUG_PERT_COL = find_col(
    drug_meta.columns,
    [
        "pert_id"
    ]
)

DRUG_NAME_COL = find_col(
    drug_meta.columns,
    [
        "pert_desc",
        "pert_iname",
        "name",
        "drug_name"
    ]
)

DRUG_MOA_COL = find_col(
    drug_meta.columns,
    [
        "MOA",
        "moa",
        "mechanism_of_action"
    ]
)

DRUG_PHASE_COL = find_col(
    drug_meta.columns,
    [
        "Phase",
        "phase",
        "clinical_phase"
    ]
)

PRED_MOA_COL = find_col(
    drug_meta.columns,
    [
        "predicted_MOA",
        "predicted_moa"
    ]
)


print(
    "Detected columns:"
)

print(
    " pert_id       =",
    DRUG_PERT_COL
)

print(
    " drug name     =",
    DRUG_NAME_COL
)

print(
    " MOA           =",
    DRUG_MOA_COL
)

print(
    " Phase         =",
    DRUG_PHASE_COL
)

print(
    " predicted MOA =",
    PRED_MOA_COL
)


# ============================================================
# Build annotation table
# ============================================================

keep_cols = [
    DRUG_PERT_COL
]

for c in [
    DRUG_NAME_COL,
    DRUG_MOA_COL,
    DRUG_PHASE_COL,
    PRED_MOA_COL
]:

    if (
        c is not None
        and
        c not in keep_cols
    ):

        keep_cols.append(c)


official_drug = (
    drug_meta[
        keep_cols
    ]
    .copy()
)


rename_map = {
    DRUG_PERT_COL:
        "pert_id"
}

if DRUG_NAME_COL:
    rename_map[
        DRUG_NAME_COL
    ] = (
        "official_drug_name"
    )

if DRUG_MOA_COL:
    rename_map[
        DRUG_MOA_COL
    ] = (
        "official_MOA"
    )

if DRUG_PHASE_COL:
    rename_map[
        DRUG_PHASE_COL
    ] = (
        "clinical_phase"
    )

if PRED_MOA_COL:
    rename_map[
        PRED_MOA_COL
    ] = (
        "predicted_MOA"
    )


official_drug = (
    official_drug
    .rename(
        columns=rename_map
    )
)


# If duplicate pert_id rows exist,
# retain the first non-empty annotation combination.
official_drug = (
    official_drug
    .drop_duplicates(
        subset=[
            "pert_id"
        ]
    )
)


annotated35 = (
    primary35
    .merge(
        official_drug,
        on="pert_id",
        how="left",
        validate="one_to_one"
    )
)


# ============================================================
# Final display name
# ============================================================

BAD_NAMES = {
    "",
    "-666",
    "unknown",
    "nan",
    "none",
    "null"
}


def valid_name(x):

    if pd.isna(x):
        return None

    x = str(x).strip()

    if (
        x.lower()
        in BAD_NAMES
    ):
        return None

    return x


def final_name(row):

    official = valid_name(
        row.get(
            "official_drug_name"
        )
    )

    existing = valid_name(
        row.get(
            "drug_name"
        )
    )

    if official:
        return official

    if existing:
        return existing

    return row[
        "pert_id"
    ]


annotated35[
    "drug_name_final"
] = annotated35.apply(
    final_name,
    axis=1
)


annotated35[
    "name_resolved"
] = (
    annotated35[
        "drug_name_final"
    ]
    !=
    annotated35[
        "pert_id"
    ]
)


annotated35.to_csv(
    RESULTS /
    "Step35E_PRIMARY_DrugAnnotation.csv",
    index=False
)


print(
    "\nPrimary candidate names resolved:",
    int(
        annotated35[
            "name_resolved"
        ].sum()
    ),
    "/",
    len(
        annotated35
    )
)


print(
    "\nStill unresolved:",
    int(
        (
            ~annotated35[
                "name_resolved"
            ]
        ).sum()
    )
)


show_cols = [
    "drug_name_final",
    "pert_id",
    "best_rank_Q1",
    "best_rank_Q2",
    "best_rank_Q3",
    "primary_consensus_strength",
]

for c in [
    "official_MOA",
    "clinical_phase",
    "predicted_MOA"
]:

    if c in annotated35.columns:
        show_cols.append(c)


print()
print(
    annotated35[
        show_cols
    ]
    .head(24)
    .to_string(
        index=False
    )
)

print()
print("=" * 90)
print("Cell 22 complete")
print("=" * 90)

In [ ]:
# Cell 23 — Liver-context signature availability audit

print("=" * 90)
print("Cell 23 — HEPG2 / PHH coverage audit")
print("=" * 90)


SIG_PERT_COL = sig_colmap[
    "pert_id"
]

SIG_ID_COL = sig_colmap[
    "sig_id"
]

SIG_CELL_COL = sig_colmap[
    "cell_id"
]


sig_meta2 = sig_meta[
    [
        SIG_ID_COL,
        SIG_PERT_COL,
        SIG_CELL_COL
    ]
].copy()

sig_meta2.columns = [
    "sig_id",
    "pert_id",
    "cell_id"
]


sig_meta2[
    "cell_id"
] = (
    sig_meta2[
        "cell_id"
    ]
    .astype(str)
    .str.upper()
)


primary_ids = set(
    annotated35[
        "pert_id"
    ]
)


liver_meta = (
    sig_meta2.loc[
        sig_meta2[
            "pert_id"
        ].isin(
            primary_ids
        )
        &
        sig_meta2[
            "cell_id"
        ].isin(
            [
                "HEPG2",
                "PHH"
            ]
        )
    ]
    .copy()
)


# ============================================================
# Total available liver-context signatures
# ============================================================

coverage_rows = []


for pert_id in (
    annotated35[
        "pert_id"
    ]
):

    sub = liver_meta.loc[
        liver_meta[
            "pert_id"
        ] == pert_id
    ]

    hepg2 = sub.loc[
        sub[
            "cell_id"
        ] == "HEPG2"
    ]

    phh = sub.loc[
        sub[
            "cell_id"
        ] == "PHH"
    ]

    coverage_rows.append({
        "pert_id":
            pert_id,

        "n_HEPG2_available":
            int(
                len(hepg2)
            ),

        "n_PHH_available":
            int(
                len(phh)
            ),

        "HEPG2_available":
            bool(
                len(hepg2) > 0
            ),

        "PHH_available":
            bool(
                len(phh) > 0
            ),

        "any_liver_context_available":
            bool(
                len(sub) > 0
            ),
    })


coverage_df = pd.DataFrame(
    coverage_rows
)


# ============================================================
# Current top50-opposite liver hits
# ============================================================

top_liver_rows = []


for pert_id in (
    annotated35[
        "pert_id"
    ]
):

    sub = sig35.loc[
        (
            sig35[
                "pert_id"
            ] == pert_id
        )
        &
        (
            sig35[
                "cell_id"
            ]
            .astype(str)
            .str.upper()
            .isin(
                [
                    "HEPG2",
                    "PHH"
                ]
            )
        )
    ].copy()


    for cell in [
        "HEPG2",
        "PHH"
    ]:

        s = sub.loc[
            sub[
                "cell_id"
            ]
            .astype(str)
            .str.upper()
            ==
            cell
        ]

        top_liver_rows.append({
            "pert_id":
                pert_id,

            "cell_context":
                cell,

            "n_top50_opposite_hits":
                int(
                    len(s)
                ),

            "query_ids":
                ";".join(
                    sorted(
                        s[
                            "query_id"
                        ]
                        .dropna()
                        .astype(str)
                        .unique()
                    )
                ),

            "best_rank":
                (
                    float(
                        s[
                            "rank"
                        ].min()
                    )
                    if len(s)
                    else np.nan
                ),
        })


top_liver = pd.DataFrame(
    top_liver_rows
)


top_liver_wide = (
    top_liver
    .pivot(
        index="pert_id",
        columns="cell_context",
        values=[
            "n_top50_opposite_hits",
            "best_rank"
        ]
    )
)


top_liver_wide.columns = [
    f"{a}_{b}"
    for a, b
    in top_liver_wide.columns
]

top_liver_wide = (
    top_liver_wide
    .reset_index()
)


liver_audit = (
    annotated35
    .merge(
        coverage_df,
        on="pert_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        top_liver_wide,
        on="pert_id",
        how="left",
        validate="one_to_one"
    )
)


liver_audit.to_csv(
    RESULTS /
    "Step35E_PRIMARY_LiverContextCoverage.csv",
    index=False
)


print(
    "Primary candidates with HEPG2 data:",
    int(
        liver_audit[
            "HEPG2_available"
        ].sum()
    ),
    "/24"
)

print(
    "Primary candidates with PHH data:",
    int(
        liver_audit[
            "PHH_available"
        ].sum()
    ),
    "/24"
)

print(
    "Primary candidates with either liver context:",
    int(
        liver_audit[
            "any_liver_context_available"
        ].sum()
    ),
    "/24"
)


print(
    "\nExisting top50 opposite liver-context hits:"
)


cols = [
    "drug_name_final",
    "pert_id",
    "n_HEPG2_available",
    "n_PHH_available",
    "n_top50_opposite_hits_HEPG2",
    "best_rank_HEPG2",
    "n_top50_opposite_hits_PHH",
    "best_rank_PHH",
]


print(
    liver_audit[
        cols
    ]
    .sort_values(
        [
            "n_top50_opposite_hits_HEPG2",
            "n_HEPG2_available"
        ],
        ascending=[
            False,
            False
        ]
    )
    .to_string(
        index=False
    )
)

print()
print("=" * 90)
print("Cell 23 complete")
print("=" * 90)

In [ ]:
# Cell 24 — Step35E REPORT

print("=" * 90)
print("STEP35E — DRUG ANNOTATION / LIVER-CONTEXT REPORT")
print("=" * 90)


n_resolved = int(
    liver_audit[
        "name_resolved"
    ].sum()
)

n_unresolved = int(
    len(liver_audit)
    -
    n_resolved
)

n_hepg2 = int(
    liver_audit[
        "HEPG2_available"
    ].sum()
)

n_phh = int(
    liver_audit[
        "PHH_available"
    ].sum()
)

n_any = int(
    liver_audit[
        "any_liver_context_available"
    ].sum()
)


print(
    "\nPrimary convergent perturbations:",
    len(
        liver_audit
    )
)

print(
    "Drug names resolved:",
    n_resolved
)

print(
    "Still unresolved:",
    n_unresolved
)

print(
    "\nHEPG2 data available:",
    f"{n_hepg2}/24"
)

print(
    "PHH data available:",
    f"{n_phh}/24"
)

print(
    "Any liver-context data:",
    f"{n_any}/24"
)


print(
    "\nTop primary candidates after annotation "
    "(NOT a therapeutic ranking):\n"
)


show = [
    "drug_name_final",
    "pert_id",
    "best_rank_Q1",
    "best_rank_Q2",
    "best_rank_Q3",
    "primary_consensus_strength",
    "n_signatures_Q1",
    "n_signatures_Q2",
    "n_signatures_Q3",
    "n_cell_lines_Q1",
    "n_cell_lines_Q2",
    "n_cell_lines_Q3",
    "n_HEPG2_available",
    "n_PHH_available",
]

for c in [
    "official_MOA",
    "clinical_phase"
]:

    if c in liver_audit.columns:
        show.append(c)


print(
    liver_audit[
        show
    ]
    .sort_values(
        [
            "primary_consensus_strength"
        ],
        ascending=False
    )
    .head(20)
    .to_string(
        index=False
    )
)


report35e = {
    "status":
        "STEP35E_ANNOTATION_AND_LIVER_COVERAGE_COMPLETE",

    "primary_candidates":
        24,

    "resolved_names":
        n_resolved,

    "unresolved_names":
        n_unresolved,

    "HEPG2_available":
        n_hepg2,

    "PHH_available":
        n_phh,

    "any_liver_context_available":
        n_any,

    "liver_context_interpretation":
        (
            "Availability audit only. Presence of HEPG2/PHH "
            "signatures is not interpreted as reversal evidence."
        ),

    "next_step":
        (
            "Retrieve candidate-specific HEPG2/PHH drug-induced "
            "up/down signatures and quantify directional reversal "
            "of the frozen Q1/Q2/Q3 gene sets."
        ),

    "claim_boundary":
        (
            "Drug annotation and liver-context coverage; "
            "not drug efficacy."
        )
}


with open(
    LOGS /
    "Step35E_FINAL_REPORT.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report35e,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("=" * 90)
print("STEP35E STATUS: COMPLETE")
print("=" * 90)

In [ ]:
# Cell 25 — Step35E liver-context handoff report

print("=" * 100)
print("STEP35E → STEP35F LIVER-CONTEXT HANDOFF")
print("=" * 100)

LIVER_PATH = (
    RESULTS /
    "Step35E_PRIMARY_LiverContextCoverage.csv"
)

x = pd.read_csv(LIVER_PATH)

# ------------------------------------------------------------
# 1. unresolved names
# ------------------------------------------------------------

unresolved = x.loc[
    ~x["name_resolved"].astype(bool),
    [
        "pert_id",
        "drug_name_final",
        "best_rank_Q1",
        "best_rank_Q2",
        "best_rank_Q3",
    ]
]

print("\n[1] Unresolved drug names")
print("-" * 100)

if len(unresolved):
    print(unresolved.to_string(index=False))
else:
    print("None")


# ------------------------------------------------------------
# 2. all candidates with liver-context availability
# ------------------------------------------------------------

liver = x.loc[
    x["any_liver_context_available"].astype(bool)
].copy()

cols = [
    "drug_name_final",
    "pert_id",
    "best_rank_Q1",
    "best_rank_Q2",
    "best_rank_Q3",
    "primary_consensus_strength",
    "n_HEPG2_available",
    "n_PHH_available",
]

optional = [
    "n_top50_opposite_hits_HEPG2",
    "best_rank_HEPG2",
    "n_top50_opposite_hits_PHH",
    "best_rank_PHH",
]

cols += [
    c
    for c in optional
    if c in liver.columns
]

print("\n[2] Primary candidates with liver-context data")
print("-" * 100)

print(
    liver[cols]
    .sort_values(
        [
            "primary_consensus_strength",
            "drug_name_final"
        ],
        ascending=[
            False,
            True
        ]
    )
    .to_string(index=False)
)


# ------------------------------------------------------------
# 3. existing HEPG2 / PHH opposite-top50 signatures
# ------------------------------------------------------------

print("\n[3] Existing liver-context opposite signatures in Q1/Q2/Q3 top50")
print("-" * 100)

sig_liver = sig35.loc[
    sig35["pert_id"].isin(
        liver["pert_id"]
    )
    &
    sig35["cell_id"]
    .astype(str)
    .str.upper()
    .isin(["HEPG2", "PHH"])
].copy()

show = [
    "query_id",
    "rank",
    "score",
    "drug_label",
    "pert_id",
    "cell_id",
    "pert_time",
    "pert_dose",
    "sig_id",
]

if len(sig_liver):

    print(
        sig_liver[show]
        .sort_values(
            [
                "pert_id",
                "query_id",
                "rank"
            ]
        )
        .to_string(index=False)
    )

else:
    print("No HEPG2/PHH signatures among current top50 opposite results.")


print()
print("Liver-context candidate count:", len(liver))
print("Existing liver-context opposite-hit rows:", len(sig_liver))

print("=" * 100)
print("HANDOFF COMPLETE")
print("=" * 100)

## Step35F. Liver-Context Directional Drug Reversal

All available HEPG2 and PHH signatures for the frozen Q1/Q2-convergent perturbation
pool are retrieved directly from L1000FWD.

HEPG2 is evaluated as the primary hepatic tumor-lineage context.
PHH is retained as a secondary hepatocyte-context sensitivity analysis.

For MS3-up programs, drug-induced down-regulation supports reversal.
For differentiated/metabolic programs, drug-induced up-regulation supports restoration.

Primary reversal is defined from Core50, G2M, Mitotic Entry, and
Spindle/Chromosome programs with equal program weighting.

No drug is selected or excluded before liver-context scoring is completed.

In [ ]:
# Cell 26 — Load frozen Step35F inputs

print("=" * 100)
print("STEP35F — LIVER-CONTEXT DIRECTIONAL REVERSAL")
print("=" * 100)

import pandas as pd
import numpy as np
import json
import requests
import time
from pathlib import Path
from urllib.parse import quote


PRIMARY_PATH = (
    RESULTS /
    "Step35D_PRIMARY_ConvergentPerturbations.csv"
)

ANNOTATION_PATH = (
    RESULTS /
    "Step35E_PRIMARY_DrugAnnotation.csv"
)

SIG_META_PATH = (
    REFERENCE /
    "L1000FWD_CD_signature_metadata.csv"
)

GENESET_PATH = (
    REFERENCE /
    "Step35B_FROZEN_DRUG_REVERSAL_GENESETS.csv"
)


for p in [
    PRIMARY_PATH,
    ANNOTATION_PATH,
    SIG_META_PATH,
    GENESET_PATH,
]:
    assert p.exists(), p


primary35 = pd.read_csv(
    PRIMARY_PATH
)

annotation35 = pd.read_csv(
    ANNOTATION_PATH
)

sig_meta35 = pd.read_csv(
    SIG_META_PATH
)

geneset35 = pd.read_csv(
    GENESET_PATH
)


assert len(primary35) == 24


EXPECTED_SIGNATURES = [
    "MS3_Core50",
    "G2M_Independent",
    "Mitotic_Entry_Independent",
    "Spindle_Chromosome_Independent",
    "Hepatic_Differentiation",
    "Bile_Acid_Metabolism",
]


assert set(
    EXPECTED_SIGNATURES
).issubset(
    set(
        geneset35["signature"]
    )
)


FROZEN = {}

for sig in EXPECTED_SIGNATURES:

    genes = (
        geneset35.loc[
            geneset35[
                "signature"
            ] == sig,
            "gene"
        ]
        .astype(str)
        .str.upper()
        .tolist()
    )

    FROZEN[sig] = set(genes)


print("Frozen signatures:")

for k, v in FROZEN.items():
    print(
        f"{k:35s}",
        len(v)
    )


print()
print("=" * 100)
print("Cell 26 complete")
print("=" * 100)

In [ ]:
# Cell 27 — Build liver-context signature manifest

print("=" * 100)
print("Cell 27 — Liver-context signature manifest")
print("=" * 100)


primary_ids = set(
    primary35[
        "pert_id"
    ]
    .astype(str)
)


meta = sig_meta35.copy()

meta[
    "pert_id"
] = (
    meta[
        "pert_id"
    ]
    .astype(str)
)

meta[
    "cell_id"
] = (
    meta[
        "cell_id"
    ]
    .astype(str)
    .str.upper()
)


liver_manifest = (
    meta.loc[
        meta[
            "pert_id"
        ].isin(
            primary_ids
        )
        &
        meta[
            "cell_id"
        ].isin(
            [
                "HEPG2",
                "PHH"
            ]
        )
    ]
    .copy()
)


name_map = (
    annotation35[
        [
            "pert_id",
            "drug_name_final"
        ]
    ]
    .drop_duplicates(
        "pert_id"
    )
)


liver_manifest = (
    liver_manifest
    .merge(
        name_map,
        on="pert_id",
        how="left",
        validate="many_to_one"
    )
)


liver_manifest = (
    liver_manifest
    .sort_values(
        [
            "cell_id",
            "drug_name_final",
            "pert_id",
            "pert_time",
            "pert_dose",
            "sig_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


liver_manifest.to_csv(
    RESULTS /
    "Step35F_LiverSignature_Manifest.csv",
    index=False
)


print(
    "Total liver-context signatures:",
    len(liver_manifest)
)

print(
    "Unique perturbations:",
    liver_manifest[
        "pert_id"
    ].nunique()
)

print()

print(
    liver_manifest.groupby(
        "cell_id"
    ).agg(
        n_signatures=(
            "sig_id",
            "nunique"
        ),
        n_perturbations=(
            "pert_id",
            "nunique"
        )
    ).to_string()
)


assert (
    liver_manifest[
        "pert_id"
    ].nunique()
    == 18
)


print()
print("=" * 100)
print("Cell 27 complete")
print("=" * 100)

In [ ]:
# Cell 28 — Retrieve liver-context drug-induced gene signatures

print("=" * 100)
print("Cell 28 — Retrieve L1000FWD signature gene sets")
print("=" * 100)


L1000FWD_URL = (
    "https://maayanlab.cloud/l1000fwd/"
)

CACHE_PATH = (
    DATA /
    "Step35F_LiverSignature_API_Cache.json"
)


# ------------------------------------------------------------
# Load existing cache
# ------------------------------------------------------------

if CACHE_PATH.exists():

    with open(
        CACHE_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        cache = json.load(f)

else:

    cache = {}


session = requests.Session()

failures = []


sig_ids = (
    liver_manifest[
        "sig_id"
    ]
    .drop_duplicates()
    .tolist()
)


print(
    "Unique liver signatures required:",
    len(sig_ids)
)

print(
    "Already cached:",
    sum(
        sid in cache
        for sid in sig_ids
    )
)


for i, sig_id in enumerate(
    sig_ids,
    start=1
):

    if sig_id in cache:

        continue


    url = (
        L1000FWD_URL
        +
        "sig/"
        +
        quote(
            str(sig_id),
            safe=""
        )
    )


    success = False

    for attempt in range(
        1,
        6
    ):

        try:

            r = session.get(
                url,
                timeout=120
            )

            if (
                r.status_code
                == 200
            ):

                obj = r.json()

                if (
                    "up_genes"
                    in obj
                    and
                    "down_genes"
                    in obj
                ):

                    cache[
                        sig_id
                    ] = obj

                    success = True

                    break

        except Exception:

            pass

        time.sleep(
            attempt * 2
        )


    if not success:

        failures.append(
            sig_id
        )

    # save after every query for resume safety
    with open(
        CACHE_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            cache,
            f,
            indent=2,
            ensure_ascii=False
        )


    if (
        i % 10 == 0
        or
        i == len(sig_ids)
    ):

        print(
            f"{i}/{len(sig_ids)} "
            f"| cached={len(cache)} "
            f"| failures={len(failures)}"
        )


failure_df = pd.DataFrame({
    "sig_id":
        failures
})

failure_df.to_csv(
    LOGS /
    "Step35F_LiverSignature_API_Failures.csv",
    index=False
)


print()
print(
    "Required:",
    len(sig_ids)
)

print(
    "Retrieved:",
    sum(
        sid in cache
        for sid in sig_ids
    )
)

print(
    "Failures:",
    len(failures)
)


print()
print("=" * 100)
print("Cell 28 complete")
print("=" * 100)

In [ ]:
# Cell 29 — Score directional reversal per liver signature

print("=" * 100)
print("Cell 29 — Signature-level directional reversal scoring")
print("=" * 100)


def score_program(
    program_genes,
    drug_up,
    drug_down,
    disease_direction
):

    P = set(
        program_genes
    )

    U = set(
        drug_up
    )

    D = set(
        drug_down
    )

    n = len(P)

    assert n > 0


    if disease_direction == "UP":

        # disease program is high;
        # drug DOWN is desired
        desired = (
            P & D
        )

        wrong = (
            P & U
        )

    elif disease_direction == "DOWN":

        # disease program is low;
        # drug UP is desired
        desired = (
            P & U
        )

        wrong = (
            P & D
        )

    else:

        raise ValueError(
            disease_direction
        )


    return {
        "desired_hits":
            len(desired),

        "wrong_hits":
            len(wrong),

        "desired_genes":
            ";".join(
                sorted(
                    desired
                )
            ),

        "wrong_genes":
            ";".join(
                sorted(
                    wrong
                )
            ),

        "net_reversal":
            (
                len(desired)
                -
                len(wrong)
            )
            /
            n,

        "coverage":
            (
                len(desired)
                +
                len(wrong)
            )
            /
            n,
    }


PROGRAM_DIRECTIONS = {
    "MS3_Core50":
        "UP",

    "G2M_Independent":
        "UP",

    "Mitotic_Entry_Independent":
        "UP",

    "Spindle_Chromosome_Independent":
        "UP",

    "Hepatic_Differentiation":
        "DOWN",

    "Bile_Acid_Metabolism":
        "DOWN",
}


rows = []


for _, meta_row in (
    liver_manifest
    .drop_duplicates(
        "sig_id"
    )
    .iterrows()
):

    sig_id = meta_row[
        "sig_id"
    ]

    if sig_id not in cache:
        continue


    obj = cache[
        sig_id
    ]


    drug_up = {
        str(x).upper()
        for x in obj.get(
            "up_genes",
            []
        )
    }

    drug_down = {
        str(x).upper()
        for x in obj.get(
            "down_genes",
            []
        )
    }


    out = {
        "sig_id":
            sig_id,

        "pert_id":
            meta_row[
                "pert_id"
            ],

        "drug_name":
            meta_row[
                "drug_name_final"
            ],

        "cell_id":
            meta_row[
                "cell_id"
            ],

        "pert_time":
            meta_row[
                "pert_time"
            ],

        "pert_dose":
            meta_row[
                "pert_dose"
            ],

        "n_drug_up_genes":
            len(
                drug_up
            ),

        "n_drug_down_genes":
            len(
                drug_down
            ),
    }


    program_net = {}


    for program in EXPECTED_SIGNATURES:

        res = score_program(
            FROZEN[
                program
            ],
            drug_up,
            drug_down,
            PROGRAM_DIRECTIONS[
                program
            ]
        )


        prefix = program


        out[
            f"{prefix}_desired_hits"
        ] = res[
            "desired_hits"
        ]

        out[
            f"{prefix}_wrong_hits"
        ] = res[
            "wrong_hits"
        ]

        out[
            f"{prefix}_net"
        ] = res[
            "net_reversal"
        ]

        out[
            f"{prefix}_coverage"
        ] = res[
            "coverage"
        ]

        out[
            f"{prefix}_desired_genes"
        ] = res[
            "desired_genes"
        ]

        out[
            f"{prefix}_wrong_genes"
        ] = res[
            "wrong_genes"
        ]


        program_net[
            program
        ] = res[
            "net_reversal"
        ]


    # ========================================================
    # Frozen formal endpoints
    # ========================================================

    out[
        "Q1_Core50_Reversal"
    ] = (
        program_net[
            "MS3_Core50"
        ]
    )


    out[
        "Q2_Mitotic_Reversal"
    ] = np.mean([
        program_net[
            "G2M_Independent"
        ],
        program_net[
            "Mitotic_Entry_Independent"
        ],
        program_net[
            "Spindle_Chromosome_Independent"
        ],
    ])


    # Equal weight to all four PRIMARY programs
    out[
        "Primary_Reversal"
    ] = np.mean([
        program_net[
            "MS3_Core50"
        ],
        program_net[
            "G2M_Independent"
        ],
        program_net[
            "Mitotic_Entry_Independent"
        ],
        program_net[
            "Spindle_Chromosome_Independent"
        ],
    ])


    out[
        "Secondary_Differentiation_Reversal"
    ] = np.mean([
        program_net[
            "Hepatic_Differentiation"
        ],
        program_net[
            "Bile_Acid_Metabolism"
        ],
    ])


    # Secondary Q3 geometry/signature logic
    out[
        "Q3_Bidirectional_Reversal"
    ] = np.mean([
        program_net[
            "MS3_Core50"
        ],
        program_net[
            "Hepatic_Differentiation"
        ],
        program_net[
            "Bile_Acid_Metabolism"
        ],
    ])


    out[
        "Primary_Programs_Positive"
    ] = int(
        sum(
            x > 0
            for x in [
                program_net[
                    "MS3_Core50"
                ],
                program_net[
                    "G2M_Independent"
                ],
                program_net[
                    "Mitotic_Entry_Independent"
                ],
                program_net[
                    "Spindle_Chromosome_Independent"
                ],
            ]
        )
    )


    rows.append(
        out
    )


signature_scores = pd.DataFrame(
    rows
)


signature_scores.to_csv(
    RESULTS /
    "Step35F_LiverSignature_DirectionalReversal.csv",
    index=False
)


print(
    "Scored liver signatures:",
    len(
        signature_scores
    )
)


print()

print(
    signature_scores.groupby(
        "cell_id"
    )[
        "sig_id"
    ]
    .nunique()
    .to_string()
)


print()
print("=" * 100)
print("Cell 29 complete")
print("=" * 100)

In [ ]:
# Cell 30 — Candidate-level liver-context aggregation

print("=" * 100)
print("Cell 30 — Candidate-level liver-context aggregation")
print("=" * 100)


agg_rows = []


for (
    pert_id,
    drug_name,
    cell_id
), sub in signature_scores.groupby(
    [
        "pert_id",
        "drug_name",
        "cell_id"
    ]
):

    primary_values = (
        sub[
            "Primary_Reversal"
        ]
        .astype(float)
    )


    agg_rows.append({
        "pert_id":
            pert_id,

        "drug_name":
            drug_name,

        "cell_id":
            cell_id,

        "n_signatures":
            int(
                len(sub)
            ),

        "median_primary_reversal":
            float(
                primary_values.median()
            ),

        "mean_primary_reversal":
            float(
                primary_values.mean()
            ),

        "min_primary_reversal":
            float(
                primary_values.min()
            ),

        "max_primary_reversal":
            float(
                primary_values.max()
            ),

        "n_primary_positive":
            int(
                (
                    primary_values
                    >
                    0
                ).sum()
            ),

        "fraction_primary_positive":
            float(
                (
                    primary_values
                    >
                    0
                ).mean()
            ),

        "median_Q1_Core50":
            float(
                sub[
                    "Q1_Core50_Reversal"
                ].median()
            ),

        "median_Q2_Mitotic":
            float(
                sub[
                    "Q2_Mitotic_Reversal"
                ].median()
            ),

        "median_secondary_differentiation":
            float(
                sub[
                    "Secondary_Differentiation_Reversal"
                ].median()
            ),

        "median_Q3_bidirectional":
            float(
                sub[
                    "Q3_Bidirectional_Reversal"
                ].median()
            ),

        "median_primary_programs_positive":
            float(
                sub[
                    "Primary_Programs_Positive"
                ].median()
            ),

        "times":
            ";".join(
                sorted(
                    sub[
                        "pert_time"
                    ]
                    .dropna()
                    .astype(str)
                    .unique()
                )
            ),

        "doses":
            ";".join(
                sorted(
                    sub[
                        "pert_dose"
                    ]
                    .dropna()
                    .astype(str)
                    .unique()
                )
            ),
    })


context_summary = pd.DataFrame(
    agg_rows
)


context_summary.to_csv(
    RESULTS /
    "Step35F_LiverContext_CandidateSummary_Long.csv",
    index=False
)


# ============================================================
# Add global Step35D evidence
# ============================================================

global_cols = [
    "pert_id",
    "primary_consensus_strength",
    "best_rank_Q1",
    "best_rank_Q2",
    "best_rank_Q3",
    "secondary_Q3_support",
]


global_info = (
    primary35[
        global_cols
    ]
    .drop_duplicates(
        "pert_id"
    )
)


context_summary = (
    context_summary
    .merge(
        global_info,
        on="pert_id",
        how="left",
        validate="many_to_one"
    )
)


context_summary.to_csv(
    RESULTS /
    "Step35F_LiverContext_CandidateSummary.csv",
    index=False
)


print(
    "Candidate-context rows:",
    len(
        context_summary
    )
)


print()

print(
    context_summary.groupby(
        "cell_id"
    ).agg(
        n_candidates=(
            "pert_id",
            "nunique"
        ),
        n_positive_median=(
            "median_primary_reversal",
            lambda x:
                int(
                    (
                        x > 0
                    ).sum()
                )
        )
    ).to_string()
)


print()
print("=" * 100)
print("Cell 30 complete")
print("=" * 100)

In [ ]:
# Cell 31 — Step35F REPORT

print("=" * 110)
print("STEP35F — LIVER-CONTEXT DIRECTIONAL REVERSAL REPORT")
print("=" * 110)


n_total_sig = len(
    signature_scores
)

n_hepg2_sig = int(
    (
        signature_scores[
            "cell_id"
        ]
        ==
        "HEPG2"
    ).sum()
)

n_phh_sig = int(
    (
        signature_scores[
            "cell_id"
        ]
        ==
        "PHH"
    ).sum()
)


hepg2 = (
    context_summary.loc[
        context_summary[
            "cell_id"
        ] == "HEPG2"
    ]
    .copy()
)


phh = (
    context_summary.loc[
        context_summary[
            "cell_id"
        ] == "PHH"
    ]
    .copy()
)


print(
    "\nLiver signatures scored:",
    n_total_sig
)

print(
    "HEPG2 signatures:",
    n_hepg2_sig
)

print(
    "PHH signatures:",
    n_phh_sig
)

print(
    "\nHEPG2 candidates:",
    len(
        hepg2
    )
)

print(
    "HEPG2 candidates with positive median primary reversal:",
    int(
        (
            hepg2[
                "median_primary_reversal"
            ]
            >
            0
        ).sum()
    )
)

print(
    "\nPHH candidates:",
    len(
        phh
    )
)

print(
    "PHH candidates with positive median primary reversal:",
    int(
        (
            phh[
                "median_primary_reversal"
            ]
            >
            0
        ).sum()
    )
)


SHOW = [
    "drug_name",
    "pert_id",
    "n_signatures",
    "median_primary_reversal",
    "fraction_primary_positive",
    "median_Q1_Core50",
    "median_Q2_Mitotic",
    "median_secondary_differentiation",
    "median_Q3_bidirectional",
    "median_primary_programs_positive",
    "primary_consensus_strength",
    "best_rank_Q1",
    "best_rank_Q2",
    "best_rank_Q3",
]


print(
    "\n[PRIMARY] HEPG2 directional reversal"
)

print("-" * 110)

if len(hepg2):

    print(
        hepg2[
            SHOW
        ]
        .sort_values(
            [
                "median_primary_reversal",
                "fraction_primary_positive",
                "primary_consensus_strength"
            ],
            ascending=[
                False,
                False,
                False
            ]
        )
        .to_string(
            index=False
        )
    )


print(
    "\n[SECONDARY] PHH directional reversal"
)

print("-" * 110)

if len(phh):

    print(
        phh[
            SHOW
        ]
        .sort_values(
            [
                "median_primary_reversal",
                "fraction_primary_positive",
                "primary_consensus_strength"
            ],
            ascending=[
                False,
                False,
                False
            ]
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# Formal descriptive support table
# ============================================================

hepg2_support = (
    hepg2.loc[
        hepg2[
            "median_primary_reversal"
        ] > 0
    ]
    .copy()
)


hepg2_support.to_csv(
    RESULTS /
    "Step35F_HEPG2_PositivePrimaryReversal.csv",
    index=False
)


report35f = {
    "status":
        "STEP35F_LIVER_CONTEXT_REVERSAL_COMPLETE",

    "primary_context":
        "HEPG2",

    "secondary_context":
        "PHH",

    "signature_level_metric":
        (
            "(desired-direction hits - wrong-direction hits) "
            "/ frozen program size"
        ),

    "primary_programs": [
        "MS3_Core50",
        "G2M_Independent",
        "Mitotic_Entry_Independent",
        "Spindle_Chromosome_Independent"
    ],

    "secondary_programs": [
        "Hepatic_Differentiation",
        "Bile_Acid_Metabolism"
    ],

    "liver_signatures_scored":
        int(
            n_total_sig
        ),

    "HEPG2_signatures":
        n_hepg2_sig,

    "PHH_signatures":
        n_phh_sig,

    "HEPG2_candidates":
        int(
            len(
                hepg2
            )
        ),

    "HEPG2_positive_median_primary_reversal":
        int(
            len(
                hepg2_support
            )
        ),

    "PHH_candidates":
        int(
            len(
                phh
            )
        ),

    "PHH_positive_median_primary_reversal":
        int(
            (
                phh[
                    "median_primary_reversal"
                ] > 0
            ).sum()
        ),

    "Q3_in_primary_score":
        False,

    "claim_boundary":
        (
            "Directional transcriptomic reversal in available "
            "L1000 liver-context signatures; not therapeutic "
            "efficacy or experimental HCC validation."
        )
}


with open(
    LOGS /
    "Step35F_FINAL_REPORT.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report35f,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("=" * 110)
print("STEP35F STATUS: COMPLETE")
print("=" * 110)

## Step35G. Final Drug-Reversal Evidence Integration

Final candidate nomination integrates two independent evidence layers:

1. Global L1000FWD cross-query convergence for the frozen Q1 and Q2 signatures.
2. Directional reversal in HEPG2 liver-context signatures.

HEPG2 is the primary liver-context validation layer.
PHH and Q3 differentiated/metabolic reversal are secondary supportive evidence.

Candidates are assigned to evidence tiers using pre-specified qualitative criteria.
No composite therapeutic score or efficacy ranking is constructed.

In [ ]:
# Cell 32 — Final Step35 evidence integration

print("=" * 110)
print("Cell 32 — Final drug-reversal evidence integration")
print("=" * 110)

import pandas as pd
import numpy as np
import json


GLOBAL_PATH = (
    RESULTS /
    "Step35D_PRIMARY_ConvergentPerturbations.csv"
)

ANNOT_PATH = (
    RESULTS /
    "Step35E_PRIMARY_DrugAnnotation.csv"
)

LIVER_PATH = (
    RESULTS /
    "Step35F_LiverContext_CandidateSummary.csv"
)


global35 = pd.read_csv(GLOBAL_PATH)
annot35 = pd.read_csv(ANNOT_PATH)
liver35 = pd.read_csv(LIVER_PATH)


# ------------------------------------------------------------
# HEPG2 / PHH separate
# ------------------------------------------------------------

hepg2 = (
    liver35.loc[
        liver35["cell_id"] == "HEPG2"
    ]
    .copy()
)

phh = (
    liver35.loc[
        liver35["cell_id"] == "PHH"
    ]
    .copy()
)


hepg2_cols = {
    "n_signatures":
        "HEPG2_n_signatures",

    "median_primary_reversal":
        "HEPG2_primary_reversal",

    "fraction_primary_positive":
        "HEPG2_fraction_positive",

    "median_Q1_Core50":
        "HEPG2_Core50",

    "median_Q2_Mitotic":
        "HEPG2_Mitotic",

    "median_secondary_differentiation":
        "HEPG2_Differentiation",

    "median_Q3_bidirectional":
        "HEPG2_Q3",

    "median_primary_programs_positive":
        "HEPG2_n_primary_programs_positive",
}


hepg2 = (
    hepg2[
        [
            "pert_id",
            *hepg2_cols.keys()
        ]
    ]
    .rename(
        columns=hepg2_cols
    )
)


phh_cols = {
    "n_signatures":
        "PHH_n_signatures",

    "median_primary_reversal":
        "PHH_primary_reversal",

    "fraction_primary_positive":
        "PHH_fraction_positive",

    "median_Q1_Core50":
        "PHH_Core50",

    "median_Q2_Mitotic":
        "PHH_Mitotic",

    "median_secondary_differentiation":
        "PHH_Differentiation",

    "median_Q3_bidirectional":
        "PHH_Q3",
}


phh = (
    phh[
        [
            "pert_id",
            *phh_cols.keys()
        ]
    ]
    .rename(
        columns=phh_cols
    )
)


# ------------------------------------------------------------
# Global + annotation + liver evidence
# ------------------------------------------------------------

annot_keep = [
    c
    for c in [
        "pert_id",
        "drug_name_final",
        "official_drug_name",
        "pubchem_cid",
    ]
    if c in annot35.columns
]


final35 = (
    global35
    .merge(
        annot35[
            annot_keep
        ]
        .drop_duplicates(
            "pert_id"
        ),
        on="pert_id",
        how="left"
    )
    .merge(
        hepg2,
        on="pert_id",
        how="left"
    )
    .merge(
        phh,
        on="pert_id",
        how="left"
    )
)


print(
    "Integrated primary candidates:",
    len(final35)
)

assert len(final35) == 24

print()
print("=" * 110)
print("Cell 32 complete")
print("=" * 110)

In [ ]:
# Cell 33 — Evidence-tier assignment

print("=" * 110)
print("Cell 33 — Final evidence-tier assignment")
print("=" * 110)


def assign_step35_tier(row):

    # No HEPG2 data
    if pd.isna(
        row[
            "HEPG2_primary_reversal"
        ]
    ):
        return "D_NO_HEPG2_VALIDATION"


    primary_positive = (
        row["HEPG2_primary_reversal"] > 0
    )

    core50_positive = (
        row["HEPG2_Core50"] > 0
    )

    mitotic_positive = (
        row["HEPG2_Mitotic"] > 0
    )


    # --------------------------------------------------------
    # Tier A:
    # replicated HEPG2 reversal
    # --------------------------------------------------------

    if (
        primary_positive
        and
        core50_positive
        and
        mitotic_positive
        and
        row["HEPG2_n_signatures"] >= 2
        and
        row["HEPG2_fraction_positive"] >= 0.75
    ):

        return (
            "A_ROBUST_HEPG2_REVERSAL"
        )


    # --------------------------------------------------------
    # Tier B:
    # positive primary reversal,
    # but only single HEPG2 signature
    # --------------------------------------------------------

    if (
        primary_positive
        and
        core50_positive
        and
        mitotic_positive
        and
        row["HEPG2_n_signatures"] == 1
    ):

        return (
            "B_SINGLE_SIGNATURE_HEPG2"
        )


    # --------------------------------------------------------
    # Tier C:
    # partial / weak liver evidence
    # --------------------------------------------------------

    if primary_positive:

        return (
            "C_PARTIAL_HEPG2_SUPPORT"
        )


    return (
        "D_NO_POSITIVE_HEPG2_REVERSAL"
    )


final35[
    "Step35_evidence_tier"
] = final35.apply(
    assign_step35_tier,
    axis=1
)


tier_order = {
    "A_ROBUST_HEPG2_REVERSAL":
        1,

    "B_SINGLE_SIGNATURE_HEPG2":
        2,

    "C_PARTIAL_HEPG2_SUPPORT":
        3,

    "D_NO_POSITIVE_HEPG2_REVERSAL":
        4,

    "D_NO_HEPG2_VALIDATION":
        5,
}


final35[
    "_tier_order"
] = (
    final35[
        "Step35_evidence_tier"
    ]
    .map(
        tier_order
    )
)


# Sorting is descriptive only.
# It is NOT therapeutic ranking.
final35 = (
    final35
    .sort_values(
        [
            "_tier_order",
            "HEPG2_primary_reversal",
            "primary_consensus_strength"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .drop(
        columns="_tier_order"
    )
    .reset_index(
        drop=True
    )
)


print(
    final35[
        "Step35_evidence_tier"
    ]
    .value_counts()
    .to_string()
)


final35.to_csv(
    RESULTS /
    "Step35G_FINAL_DrugReversal_EvidenceTable.csv",
    index=False
)


print()
print("=" * 110)
print("Cell 33 complete")
print("=" * 110)

In [ ]:
# Cell 34 — Formal candidate nomination

print("=" * 120)
print("STEP35G — FORMAL DRUG-REVERSAL CANDIDATE NOMINATION")
print("=" * 120)


formal_candidates = (
    final35.loc[
        final35[
            "Step35_evidence_tier"
        ].isin(
            [
                "A_ROBUST_HEPG2_REVERSAL",
                "B_SINGLE_SIGNATURE_HEPG2"
            ]
        )
    ]
    .copy()
)


SHOW = [
    "drug_name_final",
    "pert_id",
    "Step35_evidence_tier",

    "best_rank_Q1",
    "best_rank_Q2",
    "best_rank_Q3",

    "primary_consensus_strength",

    "HEPG2_n_signatures",
    "HEPG2_primary_reversal",
    "HEPG2_fraction_positive",

    "HEPG2_Core50",
    "HEPG2_Mitotic",

    "HEPG2_Differentiation",
    "HEPG2_Q3",

    "PHH_n_signatures",
    "PHH_primary_reversal",
    "PHH_fraction_positive",
]


SHOW = [
    c
    for c in SHOW
    if c in formal_candidates.columns
]


print(
    "\nFormal candidates "
    "(evidence tiers, NOT therapeutic ranking):\n"
)


print(
    formal_candidates[
        SHOW
    ]
    .to_string(
        index=False
    )
)


formal_candidates.to_csv(
    RESULTS /
    "Step35G_FORMAL_DrugReversal_Candidates.csv",
    index=False
)


print()
print(
    "Formal Tier A/B candidates:",
    len(
        formal_candidates
    )
)

print()
print("=" * 120)
print("Cell 34 complete")
print("=" * 120)

In [ ]:
# Cell 35 — Integrated Step33–35 interpretation

print("=" * 110)
print("STEP33 → STEP35 INTEGRATED INTERPRETATION")
print("=" * 110)


tierA = (
    final35.loc[
        final35[
            "Step35_evidence_tier"
        ]
        ==
        "A_ROBUST_HEPG2_REVERSAL"
    ]
    .copy()
)


tierB = (
    final35.loc[
        final35[
            "Step35_evidence_tier"
        ]
        ==
        "B_SINGLE_SIGNATURE_HEPG2"
    ]
    .copy()
)


print(
    "\nTier A robust HEPG2 candidates:"
)

if len(tierA):

    print(
        ", ".join(
            tierA[
                "drug_name_final"
            ].astype(str)
        )
    )

else:

    print("None")


print(
    "\nTier B single-signature HEPG2 candidates:"
)

if len(tierB):

    print(
        ", ".join(
            tierB[
                "drug_name_final"
            ].astype(str)
        )
    )

else:

    print("None")


print(
    "\nINTERPRETATION:"
)

print(
    """
1. Frozen MS3 Core50 and independent mitotic programs identified
   convergent opposite drug signatures in L1000FWD.

2. A subset of globally convergent perturbations retained positive
   directional reversal in HEPG2 liver-context signatures.

3. The most reproducible drug response involved suppression of the
   MS3 proliferative/mitotic architecture.

4. Restoration of hepatic differentiation / bile-acid programs was
   weaker and heterogeneous.

5. These results therefore support transcriptomic reversal of the
   MS3 high-risk state rather than complete restoration of an
   MS6-like differentiated phenotype.

6. Drug candidates represent in silico transcriptomic-reversal
   hypotheses and are not evidence of therapeutic efficacy.
"""
)


print("=" * 110)
print("Cell 35 complete")
print("=" * 110)

In [ ]:
# Cell 36 — STEP35 FINAL LOCK

print("=" * 110)
print("STEP35 — FINAL LOCK")
print("=" * 110)


tier_counts = (
    final35[
        "Step35_evidence_tier"
    ]
    .value_counts()
    .to_dict()
)


formal_names = (
    formal_candidates[
        "drug_name_final"
    ]
    .astype(str)
    .tolist()
)


lock35 = {
    "step":
        "Step35",

    "title":
        "Virtual Drug Reversal",

    "status":
        "FROZEN",

    "primary_question":
        (
            "Which small-molecule perturbations are predicted "
            "to reverse the conserved MS3 high-risk "
            "transcriptional architecture?"
        ),

    "frozen_queries": {
        "Q1":
            "MS3 Conserved Core50",

        "Q2":
            (
                "Independent G2M + Mitotic Entry + "
                "Spindle/Chromosome programs"
            ),

        "Q3":
            (
                "Secondary Core50 + differentiated/metabolic axis"
            )
    },

    "global_discovery": {
        "resource":
            "L1000FWD",

        "primary_convergent_perturbations":
            24,

        "primary_plus_Q3_support":
            22,
    },

    "liver_context": {
        "primary":
            "HEPG2",

        "secondary":
            "PHH",

        "HEPG2_candidates":
            17,

        "HEPG2_positive_primary_reversal":
            10,

        "PHH_candidates":
            5,

        "PHH_positive_primary_reversal":
            3,
    },

    "formal_evidence_tiers":
        tier_counts,

    "formal_Tier_A_B_candidates":
        formal_names,

    "primary_interpretation":
        (
            "A subset of globally convergent drug perturbations "
            "retained directional reversal of the frozen MS3 "
            "Core50 and independent mitotic programs in HEPG2."
        ),

    "secondary_interpretation":
        (
            "Differentiated/metabolic restoration was weaker and "
            "heterogeneous and does not establish complete "
            "MS6-like restoration."
        ),

    "integrated_Step33_35_claim":
        (
            "The MS3 proliferative/mitotic architecture shows "
            "convergent predicted vulnerability to both virtual "
            "regulator suppression and transcriptomic drug "
            "perturbation."
        ),

    "claim_boundary":
        (
            "In silico transcriptomic reversal only; "
            "not therapeutic efficacy, clinical benefit, "
            "experimental drug sensitivity, or causal "
            "state reprogramming."
        )
}


LOCK_PATH = (
    LOGS /
    "Step35_FINAL_LOCK.json"
)

with open(
    LOCK_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        lock35,
        f,
        indent=2,
        ensure_ascii=False
    )


print(
    "\nFormal Tier A/B candidates:"
)

for x in formal_names:
    print(" -", x)


print(
    "\nEvidence tiers:"
)

for k, v in tier_counts.items():
    print(
        f" - {k}: {v}"
    )


print()
print("Saved:")
print(LOCK_PATH)

print()
print("=" * 110)
print("STEP35 STATUS: FROZEN")
print("=" * 110)

## Step35H. Publication Figures and Tables

This module generates the frozen publication package for Step35.

No candidate is re-selected, re-ranked, or reclassified.

Figure 8 summarizes:
A. drug-reversal discovery and evidence filtering;
B. global Q1/Q2/Q3 convergence;
C. HEPG2 program-level directional reversal;
D. signature-level replication of Tier A candidates.

Tables preserve the complete candidate-level and signature-level evidence used for the final Step35 lock.

In [ ]:
# Cell 37 — Step35 publication package setup

print("=" * 100)
print("STEP35H — PUBLICATION PACKAGE")
print("=" * 100)

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import json

# ------------------------------------------------------------
# Formal publication settings
# ------------------------------------------------------------

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

FINAL_EVIDENCE = (
    RESULTS /
    "Step35G_FINAL_DrugReversal_EvidenceTable.csv"
)

FORMAL_CANDIDATES = (
    RESULTS /
    "Step35G_FORMAL_DrugReversal_Candidates.csv"
)

SIG_LEVEL = (
    RESULTS /
    "Step35F_LiverSignature_DirectionalReversal.csv"
)

LIVER_SUMMARY = (
    RESULTS /
    "Step35F_LiverContext_CandidateSummary.csv"
)

for p in [
    FINAL_EVIDENCE,
    FORMAL_CANDIDATES,
    SIG_LEVEL,
    LIVER_SUMMARY,
]:
    assert p.exists(), p


final35 = pd.read_csv(FINAL_EVIDENCE)
formal35 = pd.read_csv(FORMAL_CANDIDATES)
sig35f = pd.read_csv(SIG_LEVEL)
liver35 = pd.read_csv(LIVER_SUMMARY)


assert len(final35) == 24
assert len(formal35) == 8


TIER_A = (
    formal35.loc[
        formal35["Step35_evidence_tier"]
        == "A_ROBUST_HEPG2_REVERSAL"
    ]
    .copy()
)

TIER_B = (
    formal35.loc[
        formal35["Step35_evidence_tier"]
        == "B_SINGLE_SIGNATURE_HEPG2"
    ]
    .copy()
)

assert len(TIER_A) == 4
assert len(TIER_B) == 4


print("All primary-convergent candidates:", len(final35))
print("Formal Tier A/B candidates:", len(formal35))
print("Tier A:", len(TIER_A))
print("Tier B:", len(TIER_B))

print()
print("=" * 100)
print("Cell 37 complete")
print("=" * 100)

In [ ]:
# Cell 38 — Main Step35 candidate evidence table

print("=" * 100)
print("Cell 38 — Formal candidate table")
print("=" * 100)


TABLE_COLS = [
    "drug_name_final",
    "pert_id",
    "Step35_evidence_tier",

    "best_rank_Q1",
    "best_rank_Q2",
    "best_rank_Q3",

    "primary_consensus_strength",

    "HEPG2_n_signatures",
    "HEPG2_primary_reversal",
    "HEPG2_fraction_positive",

    "HEPG2_Core50",
    "HEPG2_Mitotic",
    "HEPG2_Differentiation",
    "HEPG2_Q3",

    "PHH_n_signatures",
    "PHH_primary_reversal",
    "PHH_fraction_positive",
]


table1 = (
    formal35[
        [
            c
            for c in TABLE_COLS
            if c in formal35.columns
        ]
    ]
    .copy()
)


table1 = table1.rename(
    columns={
        "drug_name_final":
            "Drug",

        "pert_id":
            "LINCS_pert_id",

        "Step35_evidence_tier":
            "Evidence_tier",

        "best_rank_Q1":
            "Q1_Core50_best_rank",

        "best_rank_Q2":
            "Q2_Mitotic_best_rank",

        "best_rank_Q3":
            "Q3_Bidirectional_best_rank",

        "primary_consensus_strength":
            "Global_primary_consensus",

        "HEPG2_n_signatures":
            "HEPG2_n",

        "HEPG2_primary_reversal":
            "HEPG2_primary_reversal",

        "HEPG2_fraction_positive":
            "HEPG2_positive_fraction",

        "HEPG2_Core50":
            "HEPG2_Core50_reversal",

        "HEPG2_Mitotic":
            "HEPG2_Mitotic_reversal",

        "HEPG2_Differentiation":
            "HEPG2_Differentiation_reversal",

        "HEPG2_Q3":
            "HEPG2_Q3_reversal",

        "PHH_n_signatures":
            "PHH_n",

        "PHH_primary_reversal":
            "PHH_primary_reversal",

        "PHH_fraction_positive":
            "PHH_positive_fraction",
    }
)


TABLE1_PATH = (
    RESULTS /
    "Table1_Step35_FormalDrugReversalCandidates.csv"
)

table1.to_csv(
    TABLE1_PATH,
    index=False
)


print(
    table1.to_string(
        index=False
    )
)

print()
print("Saved:")
print(TABLE1_PATH)

print()
print("=" * 100)
print("Cell 38 complete")
print("=" * 100)

In [ ]:
# Cell 39 — Step35 supplementary tables

print("=" * 100)
print("Cell 39 — Supplementary tables")
print("=" * 100)


# ------------------------------------------------------------
# S1 — all 24 globally convergent perturbations
# ------------------------------------------------------------

table_s1 = final35.copy()

TABLE_S1 = (
    RESULTS /
    "TableS1_Step35_All24_PrimaryConvergentPerturbations.csv"
)

table_s1.to_csv(
    TABLE_S1,
    index=False
)


# ------------------------------------------------------------
# S2 — all HEPG2 / PHH signature-level directional results
# ------------------------------------------------------------

table_s2 = sig35f.copy()

TABLE_S2 = (
    RESULTS /
    "TableS2_Step35_LiverSignatureLevelReversal.csv"
)

table_s2.to_csv(
    TABLE_S2,
    index=False
)


# ------------------------------------------------------------
# S3 — candidate × liver-context summary
# ------------------------------------------------------------

table_s3 = liver35.copy()

TABLE_S3 = (
    RESULTS /
    "TableS3_Step35_LiverContextCandidateSummary.csv"
)

table_s3.to_csv(
    TABLE_S3,
    index=False
)


print("Saved:")
print(TABLE_S1)
print(TABLE_S2)
print(TABLE_S3)

print()
print("Rows:")
print("Table S1:", len(table_s1))
print("Table S2:", len(table_s2))
print("Table S3:", len(table_s3))

print()
print("=" * 100)
print("Cell 39 complete")
print("=" * 100)

In [ ]:
# Cell 40 — Figure 8A: drug-reversal discovery workflow

print("=" * 100)
print("Cell 40 — Figure 8A")
print("=" * 100)


fig, ax = plt.subplots(
    figsize=(12, 3.4)
)

ax.set_xlim(
    0,
    1
)

ax.set_ylim(
    0,
    1
)

ax.axis("off")


steps = [
    (
        "Frozen MS3\nsignatures",
        "Q1 Core50\nQ2 Mitotic\nQ3 Secondary"
    ),
    (
        "L1000FWD\nopposite signatures",
        "54 unique\nperturbations"
    ),
    (
        "Primary\nconvergence",
        "24 Q1 + Q2\nperturbations"
    ),
    (
        "Liver-context\navailability",
        "18 candidates\nHEPG2 / PHH"
    ),
    (
        "HEPG2\nvalidation",
        "17 evaluated\n10 positive"
    ),
    (
        "Formal\nevidence tiers",
        "4 Tier A\n4 Tier B"
    ),
]


x_positions = np.linspace(
    0.08,
    0.92,
    len(steps)
)


for i, (
    title,
    detail
) in enumerate(steps):

    x = x_positions[i]

    ax.text(
        x,
        0.62,
        title,
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        bbox=dict(
            boxstyle="round,pad=0.55",
            facecolor="white",
            edgecolor="black",
            linewidth=1.2
        )
    )

    ax.text(
        x,
        0.27,
        detail,
        ha="center",
        va="center",
        fontsize=9
    )

    if i < len(steps) - 1:

        ax.annotate(
            "",
            xy=(
                x_positions[i + 1] - 0.055,
                0.62
            ),
            xytext=(
                x + 0.055,
                0.62
            ),
            arrowprops=dict(
                arrowstyle="->",
                lw=1.4
            )
        )


ax.set_title(
    "Drug-reversal discovery and liver-context validation",
    fontweight="bold",
    pad=12
)


FIG8A_PNG = (
    FIGURES /
    "Figure8A_Step35_DrugReversalWorkflow.png"
)

FIG8A_PDF = (
    FIGURES /
    "Figure8A_Step35_DrugReversalWorkflow.pdf"
)


fig.savefig(
    FIG8A_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIG8A_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


print("Saved:")
print(FIG8A_PNG)
print(FIG8A_PDF)

print()
print("=" * 100)
print("Cell 40 complete")
print("=" * 100)

In [ ]:
# Cell 41 — Figure 8B: global cross-query convergence

print("=" * 100)
print("Cell 41 — Figure 8B")
print("=" * 100)


plot_b = formal35.copy()


for q in [
    "Q1",
    "Q2",
    "Q3"
]:

    rank_col = f"best_rank_{q}"

    plot_b[
        f"{q}_strength"
    ] = np.where(
        plot_b[
            rank_col
        ].notna(),

        (
            51
            -
            plot_b[
                rank_col
            ]
        )
        /
        50,

        np.nan
    )


# Tier A first, then Tier B;
# within tier preserve final frozen order.
drug_order = (
    plot_b[
        "drug_name_final"
    ]
    .tolist()
)


matrix_b = (
    plot_b
    .set_index(
        "drug_name_final"
    )[
        [
            "Q1_strength",
            "Q2_strength",
            "Q3_strength"
        ]
    ]
    .loc[
        drug_order
    ]
)


fig, ax = plt.subplots(
    figsize=(5.4, 5.2)
)


im = ax.imshow(
    matrix_b.values,
    aspect="auto",
    vmin=0,
    vmax=1,
    cmap="Blues"
)


ax.set_xticks(
    range(3)
)

ax.set_xticklabels(
    [
        "Q1\nCore50",
        "Q2\nMitotic",
        "Q3\nBidirectional"
    ]
)


ax.set_yticks(
    range(
        len(
            matrix_b.index
        )
    )
)

ax.set_yticklabels(
    matrix_b.index
)


for i in range(
    matrix_b.shape[0]
):

    for j in range(
        matrix_b.shape[1]
    ):

        val = matrix_b.iloc[
            i,
            j
        ]

        if pd.notna(val):

            ax.text(
                j,
                i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=8
            )


cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

cbar.set_label(
    "Within-query rank strength"
)


ax.set_title(
    "Global transcriptomic reversal convergence",
    fontweight="bold"
)

ax.set_xlabel(
    "Frozen reversal query"
)

ax.set_ylabel(
    "Candidate perturbation"
)


fig.tight_layout()


FIG8B_PNG = (
    FIGURES /
    "Figure8B_Step35_GlobalCrossQueryConvergence.png"
)

FIG8B_PDF = (
    FIGURES /
    "Figure8B_Step35_GlobalCrossQueryConvergence.pdf"
)


fig.savefig(
    FIG8B_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIG8B_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


print("Saved:")
print(FIG8B_PNG)
print(FIG8B_PDF)

print()
print("=" * 100)
print("Cell 41 complete")
print("=" * 100)

In [ ]:
# Cell 42 — Figure 8C: HEPG2 program-level reversal

print("=" * 100)
print("Cell 42 — Figure 8C")
print("=" * 100)


hepg2_sig = (
    sig35f.loc[
        sig35f[
            "cell_id"
        ]
        ==
        "HEPG2"
    ]
    .copy()
)


formal_ids = set(
    formal35[
        "pert_id"
    ]
)


hepg2_sig = (
    hepg2_sig.loc[
        hepg2_sig[
            "pert_id"
        ].isin(
            formal_ids
        )
    ]
    .copy()
)


program_cols = {
    "MS3_Core50_net":
        "Core50",

    "G2M_Independent_net":
        "G2M",

    "Mitotic_Entry_Independent_net":
        "Mitotic entry",

    "Spindle_Chromosome_Independent_net":
        "Spindle /\nchromosome",

    "Hepatic_Differentiation_net":
        "Hepatic\ndifferentiation",

    "Bile_Acid_Metabolism_net":
        "Bile acid\nmetabolism",
}


median_program = (
    hepg2_sig
    .groupby(
        [
            "pert_id",
            "drug_name"
        ],
        as_index=False
    )[
        list(
            program_cols.keys()
        )
    ]
    .median()
)


name_order = (
    formal35[
        "drug_name_final"
    ]
    .tolist()
)


id_to_final_name = dict(
    zip(
        formal35[
            "pert_id"
        ],
        formal35[
            "drug_name_final"
        ]
    )
)


median_program[
    "drug_name_final"
] = (
    median_program[
        "pert_id"
    ]
    .map(
        id_to_final_name
    )
)


matrix_c = (
    median_program
    .set_index(
        "drug_name_final"
    )[
        list(
            program_cols.keys()
        )
    ]
    .reindex(
        name_order
    )
)


fig, ax = plt.subplots(
    figsize=(7.3, 5.2)
)


abs_max = np.nanmax(
    np.abs(
        matrix_c.values
    )
)

if not np.isfinite(abs_max) or abs_max == 0:
    abs_max = 0.1


im = ax.imshow(
    matrix_c.values,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-abs_max,
    vmax=abs_max
)


ax.set_xticks(
    range(
        len(
            program_cols
        )
    )
)

ax.set_xticklabels(
    list(
        program_cols.values()
    ),
    rotation=35,
    ha="right"
)


ax.set_yticks(
    range(
        len(
            matrix_c.index
        )
    )
)

ax.set_yticklabels(
    matrix_c.index
)


for i in range(
    matrix_c.shape[0]
):

    for j in range(
        matrix_c.shape[1]
    ):

        val = matrix_c.iloc[
            i,
            j
        ]

        if pd.notna(val):

            ax.text(
                j,
                i,
                f"{val:+.2f}",
                ha="center",
                va="center",
                fontsize=7.5
            )


cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.035,
    pad=0.03
)

cbar.set_label(
    "Median directional reversal"
)


ax.set_title(
    "HEPG2 reversal of frozen MS3 and differentiated programs",
    fontweight="bold"
)

ax.set_xlabel(
    "Frozen transcriptional program"
)

ax.set_ylabel(
    "Candidate perturbation"
)


fig.tight_layout()


FIG8C_PNG = (
    FIGURES /
    "Figure8C_Step35_HEPG2_ProgramReversal.png"
)

FIG8C_PDF = (
    FIGURES /
    "Figure8C_Step35_HEPG2_ProgramReversal.pdf"
)


fig.savefig(
    FIG8C_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIG8C_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


print("Saved:")
print(FIG8C_PNG)
print(FIG8C_PDF)

print()
print("=" * 100)
print("Cell 42 complete")
print("=" * 100)

In [ ]:
# Cell 43 — Figure 8D: Tier A signature-level replication
# with point-level values and median labels

print("=" * 100)
print("Cell 43 — Figure 8D")
print("=" * 100)

tierA_ids = (
    TIER_A[
        "pert_id"
    ]
    .tolist()
)

tierA_names = (
    TIER_A[
        "drug_name_final"
    ]
    .tolist()
)

tierA_map = dict(
    zip(
        tierA_ids,
        tierA_names
    )
)

rep = (
    sig35f.loc[
        (
            sig35f["cell_id"] == "HEPG2"
        )
        &
        (
            sig35f["pert_id"].isin(
                tierA_ids
            )
        )
    ]
    .copy()
)

rep[
    "drug_name_final"
] = (
    rep[
        "pert_id"
    ]
    .map(
        tierA_map
    )
)

fig, ax = plt.subplots(
    figsize=(7.2, 5.2)
)

for i, drug in enumerate(
    tierA_names
):

    sub = (
        rep.loc[
            rep[
                "drug_name_final"
            ] == drug
        ]
        .sort_values(
            "Primary_Reversal"
        )
        .copy()
    )

    if len(sub) == 1:
        jitter = np.array([0.0])
    else:
        jitter = np.linspace(
            -0.15,
            0.15,
            len(sub)
        )

    yvals = (
        sub[
            "Primary_Reversal"
        ]
        .to_numpy()
    )

    xvals = (
        np.full(
            len(sub),
            i,
            dtype=float
        )
        +
        jitter
    )

    # points
    ax.scatter(
        xvals,
        yvals,
        s=48,
        edgecolor="black",
        linewidth=0.6,
        zorder=3
    )

    # point-level numeric labels
    for x, y in zip(
        xvals,
        yvals
    ):

        ax.annotate(
            f"{y:.3f}",
            xy=(x, y),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=7,
            zorder=5
        )

    # median
    median_val = float(
        np.median(
            yvals
        )
    )

    ax.plot(
        [
            i - 0.24,
            i + 0.24
        ],
        [
            median_val,
            median_val
        ],
        linewidth=2.2,
        zorder=4
    )

    # median annotation
    ax.annotate(
        f"Median {median_val:.3f}",
        xy=(
            i + 0.25,
            median_val
        ),
        xytext=(3, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        fontsize=7.5,
        fontweight="bold"
    )

    # n annotation
    ax.text(
        i,
        ax.get_ylim()[0]
        if i > 0
        else 0,
        ""
    )

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_xticks(
    range(
        len(
            tierA_names
        )
    )
)

ax.set_xticklabels(
    tierA_names,
    rotation=25,
    ha="right"
)

ax.set_ylabel(
    "Primary reversal score"
)

ax.set_xlabel(
    "Tier A candidate perturbation"
)

ax.set_title(
    "Replicated HEPG2 reversal across independent drug signatures",
    fontweight="bold"
)

# sample-size annotation under each candidate
ymin, ymax = ax.get_ylim()
yrange = ymax - ymin

for i, drug in enumerate(
    tierA_names
):

    n = int(
        (
            rep[
                "drug_name_final"
            ] == drug
        ).sum()
    )

    ax.text(
        i,
        ymin + 0.02 * yrange,
        f"n={n}",
        ha="center",
        va="bottom",
        fontsize=8
    )

fig.tight_layout()

FIG8D_PNG = (
    FIGURES /
    "Figure8D_Step35_TierA_HEPG2_Replication.png"
)

FIG8D_PDF = (
    FIGURES /
    "Figure8D_Step35_TierA_HEPG2_Replication.pdf"
)

fig.savefig(
    FIG8D_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIG8D_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("Saved:")
print(FIG8D_PNG)
print(FIG8D_PDF)

print()
print("=" * 100)
print("Cell 43 complete")
print("=" * 100)

In [ ]:
# Cell 44 — Supplementary Figure:
# all 24 global candidates with numeric annotations

print("=" * 100)
print("Cell 44 — Supplementary global convergence")
print("=" * 100)

plot_s1 = final35.copy()

for q in [
    "Q1",
    "Q2",
    "Q3"
]:

    rank_col = f"best_rank_{q}"

    plot_s1[
        f"{q}_strength"
    ] = np.where(
        plot_s1[
            rank_col
        ].notna(),

        (
            51
            -
            plot_s1[
                rank_col
            ]
        )
        /
        50,

        np.nan
    )

matrix_s1 = (
    plot_s1
    .set_index(
        "drug_name_final"
    )[
        [
            "Q1_strength",
            "Q2_strength",
            "Q3_strength"
        ]
    ]
)

fig, ax = plt.subplots(
    figsize=(6.6, 10.2)
)

im = ax.imshow(
    matrix_s1.values,
    aspect="auto",
    vmin=0,
    vmax=1,
    cmap="Blues"
)

ax.set_xticks(
    range(3)
)

ax.set_xticklabels(
    [
        "Q1\nCore50",
        "Q2\nMitotic",
        "Q3\nBidirectional"
    ]
)

ax.set_yticks(
    range(
        len(
            matrix_s1.index
        )
    )
)

ax.set_yticklabels(
    matrix_s1.index,
    fontsize=8
)

# ------------------------------------------------------------
# numeric annotations
# ------------------------------------------------------------

for i in range(
    matrix_s1.shape[0]
):

    for j in range(
        matrix_s1.shape[1]
    ):

        val = (
            matrix_s1.iloc[
                i,
                j
            ]
        )

        if pd.notna(val):

            ax.text(
                j,
                i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=6.8
            )

        else:

            ax.text(
                j,
                i,
                "NA",
                ha="center",
                va="center",
                fontsize=6.5
            )

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.035,
    pad=0.03
)

cbar.set_label(
    "Within-query rank strength"
)

ax.set_title(
    "All Q1/Q2-convergent perturbations",
    fontweight="bold"
)

ax.set_xlabel(
    "Frozen drug-reversal query"
)

ax.set_ylabel(
    "Perturbation"
)

fig.tight_layout()

SUPP1_PNG = (
    FIGURES /
    "FigureS1_Step35_All24_GlobalConvergence.png"
)

SUPP1_PDF = (
    FIGURES /
    "FigureS1_Step35_All24_GlobalConvergence.pdf"
)

fig.savefig(
    SUPP1_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    SUPP1_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("Saved:")
print(SUPP1_PNG)
print(SUPP1_PDF)

print()
print("=" * 100)
print("Cell 44 complete")
print("=" * 100)

In [ ]:
# Cell 45 — Supplementary Figure:
# PHH sensitivity with numeric annotations

print("=" * 100)
print("Cell 45 — PHH sensitivity")
print("=" * 100)

phh_plot = (
    liver35.loc[
        liver35[
            "cell_id"
        ] == "PHH"
    ]
    .copy()
)

phh_plot = (
    phh_plot
    .sort_values(
        "median_primary_reversal",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

fig, ax = plt.subplots(
    figsize=(7.0, 4.8)
)

x = np.arange(
    len(
        phh_plot
    )
)

bars = ax.bar(
    x,
    phh_plot[
        "median_primary_reversal"
    ]
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_xticks(x)

ax.set_xticklabels(
    phh_plot[
        "drug_name"
    ],
    rotation=30,
    ha="right"
)

ax.set_ylabel(
    "Median primary reversal"
)

ax.set_xlabel(
    "Candidate perturbation"
)

ax.set_title(
    "PHH sensitivity analysis",
    fontweight="bold"
)

# ------------------------------------------------------------
# exact numeric annotation
# ------------------------------------------------------------

vals = (
    phh_plot[
        "median_primary_reversal"
    ]
    .to_numpy()
)

yrange = (
    max(vals.max(), 0)
    -
    min(vals.min(), 0)
)

offset = (
    0.04 * yrange
    if yrange > 0
    else 0.002
)

for bar, val in zip(
    bars,
    vals
):

    xpos = (
        bar.get_x()
        +
        bar.get_width() / 2
    )

    if val >= 0:

        ypos = val + offset
        va = "bottom"

    else:

        ypos = val - offset
        va = "top"

    ax.text(
        xpos,
        ypos,
        f"{val:+.3f}",
        ha="center",
        va=va,
        fontsize=8
    )

# ------------------------------------------------------------
# annotate n signatures
# ------------------------------------------------------------

for i, row in (
    phh_plot
    .iterrows()
):

    ax.text(
        i,
        0,
        f"n={int(row['n_signatures'])}",
        ha="center",
        va="bottom",
        fontsize=7,
        rotation=90
    )

fig.tight_layout()

SUPP2_PNG = (
    FIGURES /
    "FigureS2_Step35_PHH_Sensitivity.png"
)

SUPP2_PDF = (
    FIGURES /
    "FigureS2_Step35_PHH_Sensitivity.pdf"
)

fig.savefig(
    SUPP2_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    SUPP2_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("Saved:")
print(SUPP2_PNG)
print(SUPP2_PDF)

print()
print("=" * 100)
print("Cell 45 complete")
print("=" * 100)

In [ ]:
# Cell 46 — Step35 publication package FINAL REPORT

print("=" * 110)
print("STEP35H — PUBLICATION PACKAGE FINAL REPORT")
print("=" * 110)


figure_files = [
    FIG8A_PNG,
    FIG8A_PDF,

    FIG8B_PNG,
    FIG8B_PDF,

    FIG8C_PNG,
    FIG8C_PDF,

    FIG8D_PNG,
    FIG8D_PDF,

    SUPP1_PNG,
    SUPP1_PDF,

    SUPP2_PNG,
    SUPP2_PDF,
]


table_files = [
    TABLE1_PATH,
    TABLE_S1,
    TABLE_S2,
    TABLE_S3,
]


manifest_rows = []


for p in figure_files:

    assert p.exists(), p

    manifest_rows.append({
        "category":
            "Figure",

        "file":
            p.name,

        "path":
            str(p),

        "size_bytes":
            p.stat().st_size
    })


for p in table_files:

    assert p.exists(), p

    manifest_rows.append({
        "category":
            "Table",

        "file":
            p.name,

        "path":
            str(p),

        "size_bytes":
            p.stat().st_size
    })


manifest = pd.DataFrame(
    manifest_rows
)


MANIFEST_PATH = (
    RESULTS /
    "Step35H_PublicationPackage_Manifest.csv"
)

manifest.to_csv(
    MANIFEST_PATH,
    index=False
)


report35h = {
    "status":
        "STEP35_PUBLICATION_PACKAGE_COMPLETE",

    "main_figure":
        {
            "Figure8A":
                "Drug-reversal discovery workflow",

            "Figure8B":
                "Global Q1/Q2/Q3 transcriptomic convergence",

            "Figure8C":
                "HEPG2 program-level directional reversal",

            "Figure8D":
                "Tier A HEPG2 signature-level replication",
        },

    "supplementary_figures": {
        "FigureS1":
            "All 24 primary-convergent perturbations",

        "FigureS2":
            "PHH sensitivity analysis",
    },

    "tables": {
        "Table1":
            "Eight formal Tier A/B drug-reversal candidates",

        "TableS1":
            "All 24 primary-convergent perturbations",

        "TableS2":
            "Liver-context signature-level directional reversal",

        "TableS3":
            "Candidate-level HEPG2/PHH summary",
    },

    "formal_candidates":
        formal35[
            "drug_name_final"
        ]
        .astype(str)
        .tolist(),

    "Tier_A":
        TIER_A[
            "drug_name_final"
        ]
        .astype(str)
        .tolist(),

    "Tier_B":
        TIER_B[
            "drug_name_final"
        ]
        .astype(str)
        .tolist(),

    "claim_boundary":
        (
            "Evidence tiers describe transcriptomic reversal support "
            "and are not therapeutic rankings."
        ),
}


REPORT_PATH = (
    LOGS /
    "Step35H_PUBLICATION_PACKAGE_REPORT.json"
)

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report35h,
        f,
        indent=2,
        ensure_ascii=False
    )


print("\nMAIN FIGURE 8")
print(" A — Drug-reversal workflow")
print(" B — Global cross-query convergence")
print(" C — HEPG2 program-level reversal")
print(" D — Tier A signature-level replication")

print("\nSUPPLEMENTARY FIGURES")
print(" S1 — All 24 global candidates")
print(" S2 — PHH sensitivity")

print("\nTABLES")
print(" Table1 — Formal Tier A/B candidates")
print(" TableS1 — All 24 convergent perturbations")
print(" TableS2 — Signature-level liver-context results")
print(" TableS3 — Candidate-level liver-context summary")

print("\nTier A:")
for x in TIER_A["drug_name_final"]:
    print(" -", x)

print("\nTier B:")
for x in TIER_B["drug_name_final"]:
    print(" -", x)

print("\nSaved:")
print(MANIFEST_PATH)
print(REPORT_PATH)

print()
print("=" * 110)
print("STEP35H STATUS: COMPLETE")
print("STEP35 ANALYSIS + PUBLICATION PACKAGE: COMPLETE")
print("=" * 110)